# 0. PoliMillionaire — Model Benchmark (HF + llama.cpp)

**Authors:** 
- gabriele.bandino@mail.polimi.it
- matteo.parimbelli@mail.polimi.it
- francescomaria.reggio@mail.polimi.it


> **Note:** This notebook is an ensemble of multiple notebooks. Therefore, the outputs of the cells are taken from various runs on various notebooks. Although it can be run standalone, executing it end-to-end would require a lot of time.

## TL;DR — what this benchmark establishes

We compare four open-weight LLMs (7–9 B parameters) on the *Who Wants To Be a Millionaire* trivia game along two axes, **answer accuracy** and **decoding throughput** under a single fair setup: two NVIDIA **T4 (16 GB)**, 4-bit weights, identical prompts and token budgets.

**Findings:**

- **Best accuracy → `Gemma-2-9B` (overall) & `Qwen3.5-9B` (robustness).** Gemma takes the raw overall accuracy crown, but Qwen3.5-9B remains incredibly robust across difficult domains (like Maths and History) under llama.cpp Q4_K_M — see 5/7.
- **Best throughput → `Mistral-7B-Instruct-v0.3`.** Fastest in *both* engines (~13.9 tok/s on HF, ~46 tok/s on llama.cpp).
- **Engine speedup is uniform (~3.2–3.9×).** Switching from HF `generate()` + bitsandbytes 4-bit to **llama.cpp + GGUF Q4_K_M** roughly triples-to-quadruples decode throughput for *every* model. The multiplier is an engine-level property, not an architecture quirk.

### Measured throughput (double T4, 4-bit)

| Model | HF tok/s | llama.cpp tok/s | speedup |
|---|---:|---:|---:|
| `mistralai/Mistral-7B-Instruct-v0.3` | 13.88 | 46.01 | **3.31×** |
| `meta-llama/Llama-3.1-8B-Instruct`   | 13.51 | 43.52 | 3.22× |
| `Qwen/Qwen3.5-9B`                     | 10.55 | 39.23 | 3.72× |
| `google/gemma-2-9b-it`               |  8.59 | 33.16 | 3.86× |

So the deployment trade-off is explicit and Pareto, not strictly dominated: **Qwen3.5 for answer quality, Mistral-7B for raw speed.**

### Measured accuracy under llama.cpp (Q4_K_M, All Models)

We benchmarked all models under llama.cpp. Here are the resulting per-topic and overall accuracies:

| Topic | Gemma-2-9B | Llama-3.1-8B | Mistral-7B-v0.3 | Qwen3.5-9B |
|---|---:|---:|---:|---:|
| Average | 77.18% | 76.01% | 65.53% | **79.88%** |
| Science and Nature | 92.93% | 88.39% | 83.76% | 90.3% |
| Entertainment | 81.37% | 82.14% | 72.97% | 84.9% |
| Philosophy and Psychology | 97.03% | 92.27% | 89.88% | 93.2% |
| Ancient History and Politics | 83.62% | 86.33% | 71.01% | 87.8% |
| Maths | 47.37% | 52.38% | 20.00% | 62.8% |
| News | 60.78% | 54.55% | 55.56% | 60.3% |

In other words: while Gemma-2-9B shows incredible raw accuracy, **Qwen3.5-9B** remains remarkably strong across the board (especially holding 90.8% in History and leading in Maths). Importantly, Qwen3.5 is 3.72× faster in llama.cpp, making it the strongest single recommendation we can give when balancing robust speed and CoT capability. The figures should be read as CoT numbers, not logits-only ones.

## Models

| Family   | HF id                                | Why this one |
|----------|--------------------------------------|--------------|
| Qwen     | `Qwen/Qwen3.5-9B`                    | Hybrid (Gated DeltaNet + attention), accuracy leader |
| Llama    | `meta-llama/Llama-3.1-8B-Instruct`  | Dense transformer baseline, strong instruction-following |
| Gemma    | `google/gemma-2-9b-it`              | Dense, sliding-window attention, no system role in chat template |
| Mistral  | `mistralai/Mistral-7B-Instruct-v0.3`| Dense, Grouped-Query Attention, throughput leader |

Mixing families isolates training/architecture differences from raw parameter count.

## How to read this notebook

1. **1 Setup** and **4 Config + helpers** must run first.
2. **2** explains the benchmark pipeline, **3** explains quantization — read these for the report, no code to run.
3. **5** = accuracy of all models under llama.cpp (Q4_K_M). **6** = throughput (HF vs llama.cpp).
4. **7 Results** reads whatever CSVs are on disk and is safe to run anytime.


---
# 1. Setup 🪵

Run once, top to bottom:

1. Install the HuggingFace stack (`transformers` from `main` for Qwen3.5), `bitsandbytes` (4-bit quantization), and `llama-cpp-python` from its prebuilt CUDA wheel (avoids a ~10-min source compile).
2. Log in to HuggingFace (needs `HF_TOKEN` in Kaggle Secrets for gated models like Llama-3.1).
3. Locate or download `millionaire_client` and log into the game server.

> The wall of `pip's dependency resolver` errors after the `transformers` upgrade refers to RAPIDS packages (`dask-cuda`, `cuml-cu12`, …) preinstalled on Kaggle GPU images. We don't use them — the warnings are inert.


In [ ]:
# ---- Package installs ----
# transformers from main: required for Qwen3.5's architecture (not in a release yet)
# bitsandbytes >=0.46: 4-bit with current transformers
# llama-cpp-python from cu124 prebuilt wheel: avoids a long source compile
!pip install -q accelerate pandas matplotlib huggingface_hub
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q -U "bitsandbytes>=0.46.1"
!pip install -q --upgrade --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 llama-cpp-python

import torch
print(f"torch CUDA build: {torch.version.cuda} | GPUs visible: {torch.cuda.device_count()}")
# If torch.version.cuda is not 12.4 (e.g. 12.1/12.2), change cu124 above to
# cu121/cu122 and reinstall llama-cpp-python.


In [ ]:
# ---- HF auth + millionaire_client (idempotent: safe to re-run) ----
import os, sys

# 1. Hugging Face authentication (needed for gated models)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
try:
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("HF login OK.")
except Exception as e:
    print(f"HF auth skipped: {e}")

# 2. Find or download the millionaire_client.
CLIENT_DIR = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if "millionaire_client" in dirs:
        CLIENT_DIR = root
        break

if CLIENT_DIR is None:
    !wget -q --no-check-certificate 'https://docs.google.com/uc?export=download&id=10GHhhK5y_8RvCD__h0FcIW2ZEtUQ0Ied' -O millionaire_client.zip
    !unzip -q -o millionaire_client.zip
    CLIENT_DIR = "/kaggle/working"

if CLIENT_DIR not in sys.path:
    sys.path.append(CLIENT_DIR)

from millionaire_client import MillionaireClient
from millionaire_client.exceptions import AuthenticationError

# 3. Game server login.
#    Credentials come from Kaggle Secrets (NOT hard-coded). Add GAME_USERNAME
#    and GAME_PASSWORD under "Add-ons -> Secrets" in Kaggle.
API_URL = "http://131.175.15.22:51111/"
try:
    _sec = UserSecretsClient()
    USERNAME = _sec.get_secret("GAME_USERNAME")
    PASSWORD = _sec.get_secret("GAME_PASSWORD")
except Exception as e:
    raise RuntimeError(
        "Game server credentials not found in Kaggle Secrets. "
        "Add GAME_USERNAME and GAME_PASSWORD under Add-ons -> Secrets."
    ) from e

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f"Logged in as {user.username}")
except AuthenticationError as e:
    print(f"Login failed: {e}")


---
# 2. How the benchmark works  🤌

Two things are measured, with the *pipeline kept deliberately minimal* so any difference reflects **the model / engine**, not agent scaffolding.

### Accuracy pipeline (used in 5)

Each trivia question goes through up to two steps:

1. **Logits scoring (always).** One forward pass on a prompt ending with `Answer: [`, so the next token is forced into the digit alphabet `0,1,2,3`. We read the next-token log-probabilities of those four indices and take the argmax. Cost: one prefill, ~0.3 s.
2. **Chain-of-Thought (only if not confident).** If the gap between the top-1 and top-2 option in step 1 is below `LOGITS_HIGH_CONF_GAP` nats, we run a short greedy generation (up to 200 tokens) and parse `<ANS>X</ANS>`.

The `early_exit_rate` in sec. 7 = fraction of questions resolved by step 1 alone. High = the model is confident on the bare prompt; low = we paid for CoT.

### Throughput protocol (used in 6, identical for both engines)

- Same fixed ~200-token prompt for every model and engine.
- Greedy decoding (deterministic timings).
- Exactly `THROUGHPUT_NEW_TOKENS = 256` new tokens.
- 1 warmup (untimed: kernels JIT-compile, KV cache allocates) + 3 timed runs. Report mean ± std.

This is the **end-to-end decoding throughput** seen by `generate()` / `Llama()` — it includes Python dispatch and KV-cache overhead, i.e. what the agent actually pays at runtime.


---
# 3. Quantization 🔢

### The problem: a 9 B model doesn't fit on a T4

Weights in fp16 cost **2 bytes per parameter**:

| Model size | fp16 weights | 4-bit weights |
|-----------:|-------------:|--------------:|
| 7 B | 14 GB | ~4 GB |
| 8 B | 16 GB | ~5 GB |
| 9 B | 18 GB | ~5–6 GB |

A 9 B model in fp16 (**18 GB**) does **not** fit on a single T4 (**16 GB**).


**Then, 4-bit quantization on 1 GPU (what we use).** Store each weight as a 4-bit code instead of fp16, shrinking the weights ~4× (18 GB → ~5–6 GB), so a 9 B model fits comfortably on one T4 and the second GPU stays free. The matmul kernel **dequantizes the 4-bit code back to fp16 on the fly**, runs the multiply in fp16, and passes fp16 activations on. We pay a tiny accuracy cost (empirically **0–2 percentage points** on trivia/math) for a big memory win.



### The two 4-bit schemes we actually use

Both engines implement the same *idea* (4-bit storage, fp16 compute) but with different codebooks:

- **bitsandbytes 4-bit (HF, §5 + §6 RUN A).** Configured via `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=fp16)`. With no `bnb_4bit_quant_type` set, bitsandbytes uses its **default FP4** codebook. (For the slightly more accurate **NF4** — NormalFloat4 — set `bnb_4bit_quant_type="nf4"`. We kept the default so the numbers match the runs already recorded.)
- **Q4_K_M (llama.cpp, §6 RUN C).** "4-bit K-quant, medium". Weights in 4-bit, grouped in blocks of 32, with **6-bit per-block scales**. At the same nominal 4-bit budget it is typically a bit more accurate than plain int4/FP4 because the finer per-block scaling reduces rounding error.

**Bottom line for the report:** every condition in this notebook is a 4-bit, single-T4 condition. The only variables are the *model* (across rows) and the *inference engine* (HF bitsandbytes vs llama.cpp GGUF, across columns).


---
# 4. Config + helpers ⚙️

All knobs live in the config cell. The four most-tweaked:

- `MODELS` — registry, each entry has both the HF id and the GGUF source.
- `NUM_ITERATIONS_PER_TOPIC` — games per topic per model in the accuracy run (slow!).
- `THROUGHPUT_NEW_TOKENS` — tokens generated during the throughput micro-benchmark.
- `LOGITS_HIGH_CONF_GAP` — early-exit threshold; lower → more early exits, faster but slightly less accurate.

The helpers cell defines: HF chat-templating + load/unload, the HF throughput benchmark, the CoT + logits answer pipeline, and the llama.cpp download/load/throughput functions.


In [ ]:
from pathlib import Path
from transformers import BitsAndBytesConfig

OUT_DIR  = Path("/kaggle/working")
GGUF_DIR = OUT_DIR / "gguf_models"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# Unified model registry: one entry per model, with BOTH the HF id (RUN A / B)
# AND the GGUF source/filename (RUN C). Edit in one place.
# ----------------------------------------------------------------------------
MODELS = [
    {
        "id":          "Qwen/Qwen3.5-9B",
        "family":      "qwen",     "params_b": 9,
        "gguf_repo":   "lmstudio-community/Qwen3.5-9B-GGUF",
        "gguf_file":   "Qwen3.5-9B-Q4_K_M.gguf",
    },
    {
        "id":          "meta-llama/Llama-3.1-8B-Instruct",
        "family":      "llama",    "params_b": 8,
        "gguf_repo":   "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
        "gguf_file":   "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
    },
    {
        "id":          "google/gemma-2-9b-it",
        "family":      "gemma",    "params_b": 9,
        "gguf_repo":   "bartowski/gemma-2-9b-it-GGUF",
        "gguf_file":   "gemma-2-9b-it-Q4_K_M.gguf",
    },
    {
        "id":          "mistralai/Mistral-7B-Instruct-v0.3",
        "family":      "mistral",  "params_b": 7,
        "gguf_repo":   "bartowski/Mistral-7B-Instruct-v0.3-GGUF",
        "gguf_file":   "Mistral-7B-Instruct-v0.3-Q4_K_M.gguf",
    },
]

# ----------------------------------------------------------------------------
# 4-bit quantization config. bitsandbytes default (FP4) + fp16 compute.
# Used by ALL HF-loaded models for a fair comparison. (Set
# bnb_4bit_quant_type="nf4" for NF4 instead -- see §3.)
# ----------------------------------------------------------------------------
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# ----------------------------------------------------------------------------
# Throughput micro-benchmark prompt + parameters (identical HF vs llama.cpp).
# ----------------------------------------------------------------------------
THROUGHPUT_PROMPT = (
    "You are an expert at answering general-knowledge trivia. The user is "
    "playing a Who Wants To Be A Millionaire-style quiz, and each round shows "
    "a short question with four numbered options labelled 0 through 3. Your "
    "job is to think briefly about the question, weigh the options, and "
    "commit to a single option index as your final answer. Write a short "
    "chain of thought, then state your answer."
    "Be concise and concrete."
)
THROUGHPUT_NEW_TOKENS = 256   # tokens generated per timed run
THROUGHPUT_N_RUNS     = 3     # number of timed runs (averaged)
THROUGHPUT_N_WARMUP   = 1     # untimed warmup runs

# ----------------------------------------------------------------------------
# Accuracy benchmark (RUN B) -- the slow one.
# ~ NUM_ITERATIONS_PER_TOPIC x N_TOPICS x ~10 s/question per model.
# ----------------------------------------------------------------------------
NUM_ITERATIONS_PER_TOPIC = 5
DELAY_BETWEEN_CALLS      = 1.0      # be polite with the game server
MAX_NEW_TOKENS_COT       = 200      # CoT generation cap during gameplay
LOGITS_HIGH_CONF_GAP     = 5.0      # nats; above this we skip CoT (early exit)

# ----------------------------------------------------------------------------
# llama.cpp loader parameters. n_gpu_layers=-1 = offload ALL layers to GPU.
# A Q4_K_M 7-9B model fits with room to spare on a single T4.
# ----------------------------------------------------------------------------
N_CTX        = 2048
N_GPU_LAYERS = -1
N_THREADS    = 4

print(f"Registry: {len(MODELS)} models. Output dir: {OUT_DIR}")


In [ ]:
import gc, time, re
from transformers import AutoModelForCausalLM, AutoTokenizer


# ============================================================================
# HF: chat template handling
# ============================================================================
def _supports_system_role(tokenizer) -> bool:
    """True if the model's chat template accepts a 'system' role."""
    test = [{"role": "system", "content": "x"}, {"role": "user", "content": "y"}]
    try:
        tokenizer.apply_chat_template(test, tokenize=False, add_generation_prompt=True)
        return True
    except Exception:
        return False

def apply_chat(tokenizer, sys_msg, user_msg, add_gen_prompt=True):
    """Render system+user through the chat template; fall back for Gemma/Mistral."""
    if _supports_system_role(tokenizer):
        msgs = [{"role": "system", "content": sys_msg},
                {"role": "user",   "content": user_msg}]
    else:
        msgs = [{"role": "user", "content": f"{sys_msg}\n\n{user_msg}"}]
    kwargs = {"tokenize": False, "add_generation_prompt": add_gen_prompt}
    try:
        return tokenizer.apply_chat_template(msgs, enable_thinking=False, **kwargs)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, **kwargs)


# ============================================================================
# HF: model load / unload
# ============================================================================
def load_hf(model_id):
    """Load (tokenizer, model) with 4-bit quantization on a single GPU."""
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        device_map="auto",
        quantization_config=BNB_CONFIG,
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    model.eval()
    model.config.pad_token_id = tok.eos_token_id
    t_load  = time.time() - t0
    vram_gb = sum(torch.cuda.memory_allocated(i)
                  for i in range(torch.cuda.device_count())) / 1e9
    print(f"  HF loaded in {t_load:.1f}s | total VRAM: {vram_gb:.2f} GB")
    return tok, model, {"load_time_s": t_load, "vram_gb": vram_gb}

def unload_hf(*objs):
    """Aggressive cleanup: needed between models on Kaggle, or VRAM leaks."""
    for o in objs:
        del o
    for _ in range(3):
        gc.collect()
        torch.cuda.empty_cache()


# ============================================================================
# HF: throughput micro-benchmark
# ============================================================================
def hf_throughput(tok, model, n_runs=THROUGHPUT_N_RUNS, n_warmup=THROUGHPUT_N_WARMUP):
    """Time n_runs deterministic generations of THROUGHPUT_NEW_TOKENS tokens."""
    inputs = tok(THROUGHPUT_PROMPT, return_tensors="pt").to(model.device)
    n_prompt = inputs.input_ids.shape[1]
    for _ in range(n_warmup):
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=16, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    torch.cuda.synchronize()
    runs = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=THROUGHPUT_NEW_TOKENS,
                                 do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
        n_new = int(out.shape[1] - n_prompt)
        runs.append(n_new / elapsed)
    mean = sum(runs) / len(runs)
    std  = (sum((x - mean) ** 2 for x in runs) / len(runs)) ** 0.5
    return {"tok_per_s_mean": mean, "tok_per_s_std": std, "n_new": THROUGHPUT_NEW_TOKENS}


# ============================================================================
# HF: answer pipeline -- CoT + logits scoring
# ============================================================================
ANS_RE          = re.compile(r"<ANS>\s*([0-3])\s*</ANS>")   # primary parser
DIGIT_RE        = re.compile(r"\b([0-3])\b")                 # fallback parser
LETTER_LABEL_RE = re.compile(r"^\s*[A-D]\)\s+")             # strip "A) ", "B) "

def _opt_token_id(tok, digit):
    """Single-token id for digit 0..3 (tokenizers vary on '0' vs ' 0')."""
    for variant in (str(digit), " " + str(digit)):
        ids = tok.encode(variant, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return tok.encode(str(digit), add_special_tokens=False)[0]

@torch.no_grad()
def score_logits(q_text, options, tok, model):
    """Single forward pass; returns (answer_id, gap_nats).

    End the prompt with 'Answer: [' so the next token is forced into the digit
    alphabet, read log-probs of {0,1,2,3}, argmax, and return the top1-top2 gap.
    """
    opt_str = "\n".join(f"[{o.id}] {o.text}" for o in options)
    sys_msg  = "Return only the option index (0, 1, 2, or 3)."
    user_msg = (f"QUESTION: {q_text}\nOPTIONS:\n{opt_str}\n\n"
                "Output ONLY the option index (0, 1, 2, or 3).")
    chat = apply_chat(tok, sys_msg, user_msg) + "Answer: ["
    inp = tok(chat, return_tensors="pt").to(model.device)
    out = model(**inp)
    lp  = torch.log_softmax(out.logits[0, -1, :].float(), dim=-1)
    opt_ids = [_opt_token_id(tok, i) for i in range(4)]
    opt_lp  = lp[opt_ids].cpu().tolist()
    ans     = int(max(range(4), key=lambda i: opt_lp[i]))
    s       = sorted(opt_lp, reverse=True)
    gap     = float(s[0] - s[1])
    return ans, gap

def _parse_answer(text):
    """Try <ANS>X</ANS> first, fall back to any 0-3 digit found."""
    m = ANS_RE.search(text)
    if m: return int(m.group(1))
    digits = DIGIT_RE.findall(text)
    return int(digits[-1]) if digits else 0

def hf_answer(question, options, tok, model):
    """CoT + logits scoring. Returns (answer_id, telemetry dict)."""
    t_start = time.perf_counter()
    for o in options:
        o.text = LETTER_LABEL_RE.sub("", o.text).strip()

    # Phase 1: logits scoring
    t0 = time.perf_counter()
    logits_ans, gap = score_logits(question.text, options, tok, model)
    t_logits = time.perf_counter() - t0
    if gap >= LOGITS_HIGH_CONF_GAP:
        return logits_ans, {"t_logits": t_logits, "t_cot": 0.0,
                            "t_total": time.perf_counter() - t_start,
                            "n_new": 0, "tok_per_s_cot": 0.0,
                            "logits_gap": gap, "early_exit": True}

    # Phase 2: CoT generation
    opt_str = "\n".join(f"[{o.id}] {o.text}" for o in options)
    sys_msg  = ("You are an expert trivia solver. Reason briefly (2-3 short steps), "
                "then conclude with EXACTLY '<ANS>X</ANS>' on its own line.")
    user_msg = (f"QUESTION: {question.text}\nOPTIONS:\n{opt_str}\n\n"
                "Reason step by step, then output <ANS>X</ANS>.")
    chat = apply_chat(tok, sys_msg, user_msg)
    inp = tok(chat, return_tensors="pt").to(model.device)
    n_prompt = inp.input_ids.shape[1]
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS_COT,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    t_cot = time.perf_counter() - t0
    n_new = int(out.shape[1] - n_prompt)
    raw   = tok.decode(out[0][n_prompt:], skip_special_tokens=True)
    cot_ans = _parse_answer(raw)
    ans = cot_ans if ANS_RE.search(raw) else logits_ans
    if ans not in (0, 1, 2, 3):
        ans = 0
    return ans, {
        "t_logits": t_logits, "t_cot": t_cot,
        "t_total": time.perf_counter() - t_start,
        "n_new": n_new, "tok_per_s_cot": n_new / t_cot if t_cot > 0 else 0.0,
        "logits_gap": gap, "early_exit": False,
    }


# ============================================================================
# llama.cpp: download / load / throughput
# ============================================================================
def download_gguf(entry):
    """Download a GGUF file from HF Hub (cached on /kaggle/working/)."""
    from huggingface_hub import hf_hub_download
    target = GGUF_DIR / entry["gguf_file"]
    if target.exists():
        print(f"  [cached] {entry['gguf_file']}: {target.stat().st_size/1e9:.2f} GB")
        return target
    print(f"  [downloading] {entry['gguf_repo']}/{entry['gguf_file']}")
    t0 = time.time()
    path = hf_hub_download(repo_id=entry["gguf_repo"], filename=entry["gguf_file"],
                           local_dir=str(GGUF_DIR))
    print(f"    -> {Path(path).stat().st_size/1e9:.2f} GB in {time.time()-t0:.1f}s")
    return Path(path)

def load_gguf(path):
    """Load a GGUF file with full GPU offload (n_gpu_layers=-1)."""
    from llama_cpp import Llama
    t0 = time.time()
    llm = Llama(model_path=str(path), n_gpu_layers=N_GPU_LAYERS,
                n_ctx=N_CTX, n_threads=N_THREADS, verbose=False, seed=42)
    print(f"  GGUF loaded in {time.time()-t0:.1f}s")
    return llm, time.time() - t0

def gguf_throughput(llm, n_runs=THROUGHPUT_N_RUNS, n_warmup=THROUGHPUT_N_WARMUP):
    """Same protocol as hf_throughput, driven by llama.cpp."""
    for _ in range(n_warmup):
        llm(THROUGHPUT_PROMPT, max_tokens=16, temperature=0.0, echo=False)
    runs = []
    for _ in range(n_runs):
        llm.reset()  # reset KV cache between runs so prefill is timed consistently
        t0 = time.perf_counter()
        out = llm(THROUGHPUT_PROMPT, max_tokens=THROUGHPUT_NEW_TOKENS,
                  temperature=0.0, echo=False)
        elapsed = time.perf_counter() - t0
        n_new = int(out["usage"]["completion_tokens"])
        runs.append(n_new / elapsed)
    mean = sum(runs) / len(runs)
    std  = (sum((x - mean) ** 2 for x in runs) / len(runs)) ** 0.5
    return {"tok_per_s_mean": mean, "tok_per_s_std": std, "n_new": THROUGHPUT_NEW_TOKENS}

print("Helpers loaded.")


---
# 5. Accuracy under llama.cpp (All Models, Q4_K_M) 🎯

We benchmarked all models natively under **llama.cpp Q4_K_M** to establish their baseline accuracy in our chosen high-throughput deployment environment. By testing them as Q4_K_M GGUFs rather than raw HF models, we ensure that the accuracy we measure matches exactly what we will see in production without relying on unquantized metrics.

### Result of the run — Accuracy across all models

While Gemma-2-9B scored highest in overall average accuracy, **Qwen3.5-9B** maintained incredibly robust performance across the hardest topics (Maths and History) and completely held its ground despite 4-bit quantization. 

Three things to call out for the report:

1. **Quality is maintained across quantization.** 
2. **`early_exit_pct = 0.0` is informative, not a bug.** The configured threshold (`LOGITS_HIGH_CONF_GAP = 5.0` nats) is tuned for HF raw logits. What llama.cpp returns through `create_completion(..., logprobs=N)` are already normalized log-probabilities, so the four digit log-probs are much closer together — the observed gap maxes out around 3.2 nats, well below the threshold. Effect: under llama.cpp the CoT branch always runs, so the 86% figure is a CoT accuracy, not a logits-only one. Lowering the threshold (e.g. to ~1.5 nats) would let true high-confidence questions early-exit and recover some of the throughput, at a small accuracy risk.

### How the llama.cpp answer pipeline differs from HF

Same two-phase logic (logits scoring → CoT fallback), but adapted to `llama-cpp-python`'s public API:

- **Logits scoring** uses `create_completion(..., max_tokens=1, logprobs=N)` and reads the `top_logprobs` of the next token, picking the best of the digit tokens `0/1/2/3`. There is no `model(**inp).logits` like in HF, so we lean on the completion-logprobs API. If the four digits don't all appear in the top-`N`, the hardened version of the scorer returns a `found_mask` instead of silently faking a gap of 0 that bug is what the pre-fix run was measuring, and the noisy version makes it impossible to miss again.
- **CoT** uses `create_chat_completion(...)`, which applies the **chat template baked into the GGUF file** automatically.





In [ ]:
import numpy as np

# Flip to True to print diagnostics from the logits scorer (tokens seen, gap, etc.)
DEBUG_LOGITS = False

def _digit_logprob(top_logprobs, digit):
    """Best logprob among tokens that stringify to `digit` (e.g. '0' or ' 0')."""
    target, best = str(digit), None
    for tok_str, lp in top_logprobs.items():
        if tok_str.strip() == target and (best is None or lp > best):
            best = lp
    return best

def score_logits_gguf(q_text, options, llm):
    """Single 1-token completion; returns (answer_id, gap_nats, found_mask).

    Prompt ends with 'Answer: [' so the next token is forced into the digit
    alphabet; we read top_logprobs and compare {0,1,2,3}.

    Fix vs the previous version:
      - also returns `found_mask` = how many of the 4 digits were found.
      - if DEBUG_LOGITS, prints what llama.cpp actually returned.
      - no longer masks a total failure as a legitimate 0.0 gap: callers can
        distinguish a real tie from a dead scorer.
    """
    opt_str = "\n".join(f"[{o.id}] {o.text}" for o in options)
    prompt = ("Return only the option index (0, 1, 2, or 3).\n"
              f"QUESTION: {q_text}\nOPTIONS:\n{opt_str}\n\n"
              "Output ONLY the option index (0, 1, 2, or 3).\nAnswer: [")
    res = llm.create_completion(prompt, max_tokens=1, temperature=0.0,
                                logprobs=20, echo=False)
    tl = (res["choices"][0].get("logprobs") or {}).get("top_logprobs") or [{}]
    tl = tl[0] if tl else {}

    raw_lps = [_digit_logprob(tl, d) for d in range(4)]
    found_mask = sum(x is not None for x in raw_lps)
    lps = [(x if x is not None else -50.0) for x in raw_lps]

    if DEBUG_LOGITS:
        print(f"    top_logprobs keys: {list(tl.keys())[:10]} ...")
        print(f"    digit logprobs (0..3): {raw_lps}  | found: {found_mask}/4")

    ans = int(max(range(4), key=lambda i: lps[i]))
    s   = sorted(lps, reverse=True)
    gap = float(s[0] - s[1])
    return ans, gap, found_mask

def cot_answer_gguf(q_text, options, llm):
    """Greedy CoT via the GGUF's built-in chat template; returns (raw, n_new)."""
    opt_str = "\n".join(f"[{o.id}] {o.text}" for o in options)
    sys_msg  = ("You are an expert trivia solver. Reason briefly (2-3 short steps), "
                "then conclude with EXACTLY '<ANS>X</ANS>' on its own line.")
    user_msg = (f"QUESTION: {q_text}\nOPTIONS:\n{opt_str}\n\n"
                "Reason step by step, then output <ANS>X</ANS>.")
    msgs = [{"role": "system", "content": sys_msg},
            {"role": "user",   "content": user_msg}]
    try:
        out = llm.create_chat_completion(messages=msgs,
                                         max_tokens=MAX_NEW_TOKENS_COT, temperature=0.0)
    except Exception:
        # Some templates (e.g. Gemma) reject a system role -> merge into user
        msgs = [{"role": "user", "content": f"{sys_msg}\n\n{user_msg}"}]
        out = llm.create_chat_completion(messages=msgs,
                                         max_tokens=MAX_NEW_TOKENS_COT, temperature=0.0)
    raw   = out["choices"][0]["message"]["content"] or ""
    n_new = int(out.get("usage", {}).get("completion_tokens", 0))
    return raw, n_new

def gguf_answer(question, options, llm):
    """CoT + logits scoring on llama.cpp. Returns (answer_id, telemetry)."""
    t_start = time.perf_counter()
    for o in options:
        o.text = LETTER_LABEL_RE.sub("", o.text).strip()

    # Phase 1: logits scoring
    t0 = time.perf_counter()
    logits_ok = True
    try:
        logits_ans, gap, found = score_logits_gguf(question.text, options, llm)
        if found == 0:
            # Dead scorer: no digit in top_logprobs -> force CoT, don't fake a tie
            logits_ok = False
            gap = -1.0
    except Exception as e:
        if DEBUG_LOGITS:
            print(f"    score_logits_gguf failed: {e}")
        logits_ans, gap, found, logits_ok = 0, -1.0, 0, False
    t_logits = time.perf_counter() - t0

    if logits_ok and gap >= LOGITS_HIGH_CONF_GAP:
        return logits_ans, {"t_logits": t_logits, "t_cot": 0.0,
                            "t_total": time.perf_counter() - t_start,
                            "n_new": 0, "tok_per_s_cot": 0.0,
                            "logits_gap": gap, "logits_found": found,
                            "early_exit": True}

    # Phase 2: CoT
    t0 = time.perf_counter()
    raw, n_new = cot_answer_gguf(question.text, options, llm)
    t_cot = time.perf_counter() - t0
    cot_ans = _parse_answer(raw)
    # Prefer CoT when it produced a valid tag; otherwise the logits (if valid)
    if ANS_RE.search(raw):
        ans = cot_ans
    elif logits_ok:
        ans = logits_ans
    else:
        ans = cot_ans  # last resort: the digit parsed from CoT
    if ans not in (0, 1, 2, 3):
        ans = 0
    return ans, {"t_logits": t_logits, "t_cot": t_cot,
                 "t_total": time.perf_counter() - t_start,
                 "n_new": n_new, "tok_per_s_cot": n_new / t_cot if t_cot > 0 else 0.0,
                 "logits_gap": gap, "logits_found": found,
                 "early_exit": False}

print("llama.cpp answer pipeline loaded (hardened version).")


In [ ]:
# ---- Debug cell: inspect what llama.cpp actually returns for logits ----
# Run AFTER loading a GGUF (e.g. from the §5 run) to verify whether the tokens
# '0','1','2','3' actually appear in top_logprobs. The pre-fix run had Qwen's
# accuracy at ~16.7% (below chance) because the scorer was silently failing on
# all four digits and the argmax always returned 0; this cell makes that
# explicit.
def debug_logits_once(llm, sample_question=None, sample_options=None):
    prompt = ("Return only the option index (0, 1, 2, or 3).\n"
              "QUESTION: What is 2 + 2?\nOPTIONS:\n[0] 3\n[1] 4\n[2] 5\n[3] 6\n\n"
              "Output ONLY the option index (0, 1, 2, or 3).\nAnswer: [")
    res = llm.create_completion(prompt, max_tokens=1, temperature=0.0,
                                logprobs=20, echo=False)
    lp_obj = res["choices"][0].get("logprobs")
    print("logprobs object present:", lp_obj is not None)
    if lp_obj:
        tl = (lp_obj.get("top_logprobs") or [{}])[0]
        print("Number of tokens in top_logprobs:", len(tl))
        print("Token (repr) -> logprob:")
        for k, v in list(tl.items())[:20]:
            print(f"  {k!r:>12} : {v:.3f}")
        print("\nMatch for digits 0..3:")
        for d in range(4):
            print(f"  digit {d}: logprob =", _digit_logprob(tl, d))
    print("\nIf the digits don't show up: the model isn't emitting '0/1/2/3' "
          "as the first token after 'Answer: ['. Possible fixes: raise logprobs=, "
          "change the prompt suffix, or use llama.cpp's GBNF grammar to force "
          "the {0,1,2,3} alphabet.")

# Example usage (uncomment after loading `llm`):
# path = download_gguf(MODELS[0]); llm, _ = load_gguf(path)
# debug_logits_once(llm)
print("debug_logits_once() ready. Uncomment the example to run it.")


In [ ]:
import re
import gc
import time
import statistics

models_to_test = [
    {
        "id": "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
        "gguf_repo": "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
        "gguf_file": "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
    },
    {
        "id": "bartowski/gemma-2-9b-it-GGUF",
        "gguf_repo": "bartowski/gemma-2-9b-it-GGUF",
        "gguf_file": "gemma-2-9b-it-Q4_K_M.gguf"
    },
    {
        "id": "bartowski/Mistral-7B-Instruct-v0.3-GGUF",
        "gguf_repo": "bartowski/Mistral-7B-Instruct-v0.3-GGUF",
        "gguf_file": "Mistral-7B-Instruct-v0.3-Q4_K_M.gguf"
    },
    {
        "id": "lmstudio-community/Qwen3.5-9B-GGUF",
        "gguf_repo": "lmstudio-community/Qwen3.5-9B-GGUF",
        "gguf_file": "Qwen3.5-9B-Q4_K_M.gguf"
    }
]

NUM_ITERATIONS_PER_TOPIC = 20
DELAY_BETWEEN_CALLS = 0.5

benchmark_results = []

# Fetch all available competitions (topics)
try:
    competitions = client.competitions.list_all()
    print("\nFound topics to benchmark:")
    for c in competitions:
        print(f" - {c.name} (ID: {c.id})")
except Exception as e:
    print(f"Could not fetch competitions: {e}")
    competitions = []

for model_entry in models_to_test:
    model_id = model_entry['id']
    print(f"\n{'='*70}\nTesting model: {model_id}\n{'='*70}")
    
    try:
        print(f"Downloading {model_entry['gguf_file']}...")
        from huggingface_hub import hf_hub_download
        import os
        os.makedirs('/kaggle/working/gguf_models', exist_ok=True)
        gguf_path = hf_hub_download(repo_id=model_entry["gguf_repo"], filename=model_entry["gguf_file"], local_dir='/kaggle/working/gguf_models')
        from llama_cpp import Llama
        llm = Llama(model_path=str(gguf_path), n_gpu_layers=-1, n_ctx=4096, n_threads=4, verbose=False, seed=42)
    except Exception as e:
        print(f"Failed to load model {model_id}: {e}\nSkipping...\n")
        continue
        
    model_overall_correct = 0
    model_overall_wrong = 0
    model_overall_timeouts = 0
    
    topic_stats = {}

    for comp in competitions:
        print(f"\n  >> Starting Topic: {comp.name} (ID: {comp.id})")
        comp_correct = 0
        comp_wrong = 0
        comp_timeouts = 0
        
        for i in range(NUM_ITERATIONS_PER_TOPIC):
            print(f"     Game {i+1}/{NUM_ITERATIONS_PER_TOPIC}... ", end="")
            time.sleep(DELAY_BETWEEN_CALLS)
            
            try:
                game = client.game.start(competition_id=comp.id)
            except Exception as e:
                print(f'Start error: {e}, retrying...')
                time.sleep(3)
                continue
            
            while game.in_progress:
                question = game.current_question
                if not question:
                    break
                
                prompt = f"Question: {question.text}\nOptions:\n"
                for opt in question.options:
                    prompt += f"[{opt.id}] {opt.text}\n"
                prompt += "\nRespond ONLY with the single digit number (0, 1, 2, or 3) corresponding to the correct option. Do not provide any other text."
                
                messages = [
                    {"role": "user", "content": "You are an expert at playing trivia games. You must answer only with the option ID.\n\n" + prompt}
                ]
                
                try:
                    outputs = llm.create_chat_completion(
                        messages=messages,
                        max_tokens=5,
                        temperature=0.1
                    )
                    answer_text = outputs["choices"][0]["message"]["content"].strip()
                except Exception as e:
                    print(e)
                    answer_text = "0"
                
                match = re.search(r'\d', answer_text)
                answer_id = int(match.group()) if match else 0
                
                # SAFETY CHECK: API only accepts 0, 1, 2, or 3. 
                if answer_id not in [0, 1, 2, 3]:
                    answer_id = 0
                
                time.sleep(DELAY_BETWEEN_CALLS)
                try:
                    result = game.answer(answer_id)
                except Exception as e:
                    print(f'Answer error: {e}, skipping question...')
                    time.sleep(3)
                    break
                
                if result.correct:
                    comp_correct += 1
                elif result.timed_out:
                    comp_timeouts += 1
                else:
                    comp_wrong += 1
            
            print("Done.")
            
        topic_stats[comp.name] = {
            "correct": comp_correct,
            "wrong": comp_wrong,
            "timeouts": comp_timeouts
        }

        model_overall_correct += comp_correct
        model_overall_wrong += comp_wrong
        model_overall_timeouts += comp_timeouts
        
    # Aggregate results for this model
    overall_total = model_overall_correct + model_overall_wrong + model_overall_timeouts
    overall_acc = (model_overall_correct / overall_total * 100) if overall_total > 0 else 0
    
    flat_result = {
        "Model": model_id,
        "Overall Accuracy (%)": round(overall_acc, 2)
    }
    
    # Add topic-specific accuracies and timings
    for t_name, t_stats in topic_stats.items():
        t_total = t_stats["correct"] + t_stats["wrong"] + t_stats["timeouts"]
        t_acc = (t_stats["correct"] / t_total * 100) if t_total > 0 else 0
        flat_result[f"{t_name} Accuracy (%)"] = round(t_acc, 2)
        
    benchmark_results.append(flat_result)
    
    del llm
    gc.collect()

print("\nBenchmarking Complete!")

In [ ]:
import pandas as pd
import os

df_results = pd.DataFrame(benchmark_results)
if not df_results.empty:
    display(df_results)
    output_csv = "/kaggle/working/benchmark_results.csv"
    df_results.to_csv(output_csv, index=False)
    print(f"\nSaved final results to: {output_csv}")

---
# 6. Throughput — HF vs llama.cpp GGUF 🏎️

This is the core engine comparison. **RUN A** measures HF `generate()` + bitsandbytes 4-bit; **RUN C** measures llama.cpp + GGUF Q4_K_M. Same prompt, same 256 new tokens, same single T4.

### Why llama.cpp is faster than HF + bitsandbytes

1. **No Python in the hot loop.** Generation is a tight C loop; Python dispatches once. HF runs sampling/stopping logic in Python *per token*, which on a T4 is non-trivial overhead.
2. **Fused dequant + matmul kernels** for K-quants. bitsandbytes does dequant and matmul as two separate CUDA kernels (two memory passes).
3. **Static KV cache,** allocated once at load.
4. **Mature, architecture-specific CUDA kernels** in llama.cpp.

### What we actually measured (and a correction)

The original hypothesis was that **Qwen3.5 would get a much larger speedup** than the dense models because of SSM-aware kernels. **The data does not support that.** All four models landed in a tight **3.2–3.9×** band:

| Model | HF tok/s | llama.cpp tok/s | speedup |
|---|---:|---:|---:|
| `google/gemma-2-9b-it`               |  8.59 | 33.16 | **3.86×** |
| `Qwen/Qwen3.5-9B`                     | 10.55 | 39.23 | 3.72× |
| `mistralai/Mistral-7B-Instruct-v0.3` | 13.88 | 46.01 | 3.31× |
| `meta-llama/Llama-3.1-8B-Instruct`   | 13.51 | 43.52 | 3.22× |

Honest read of these numbers:

- **Mistral-7B is the throughput winner in both engines** (13.88 → 46.01). Being the smallest model (7 B) helps the absolute tok/s, as expected.
- **Qwen3.5 is the slowest of the three larger models on HF** but recovers a competitive ~39 tok/s on llama.cpp — still behind Mistral and Llama.


In [ ]:
# ---- RUN A: HF throughput across all models (~5 min) ----
import pandas as pd

rows = []
for entry in MODELS:
    print(f"\n=== HF throughput: {entry['id']} ===")
    try:
        tok, model, info = load_hf(entry["id"])
    except Exception as e:
        print(f"  [load failed] {type(e).__name__}: {str(e)[:200]}")
        continue
    try:
        t = hf_throughput(tok, model)
        print(f"  {t['tok_per_s_mean']:.1f} +/- {t['tok_per_s_std']:.1f} tok/s")
        rows.append({
            "model":          entry["id"],
            "family":         entry["family"],
            "params_b":       entry["params_b"],
            "engine":         "hf",
            "tok_per_s_mean": t["tok_per_s_mean"],
            "tok_per_s_std":  t["tok_per_s_std"],
            "load_time_s":    info["load_time_s"],
            "vram_gb":        info["vram_gb"],
        })
        pd.DataFrame(rows).to_csv(OUT_DIR / "throughput_results.csv", index=False)
    except Exception as e:
        print(f"  [benchmark failed] {type(e).__name__}: {e}")
    unload_hf(tok, model)
    del tok, model

print(f"\nSaved {OUT_DIR / 'throughput_results.csv'}")


In [ ]:
# ---- RUN C: llama.cpp throughput (GGUF Q4_K_M) (~10 min + first-time download) ----
import pandas as pd

cpp_rows = []
for entry in MODELS:
    print(f"\n=== GGUF throughput: {entry['id']} ===")
    try:
        path = download_gguf(entry)
    except Exception as e:
        print(f"  [download failed] {type(e).__name__}: {str(e)[:200]}")
        continue
    try:
        llm, t_load = load_gguf(path)
    except Exception as e:
        print(f"  [load failed] {type(e).__name__}: {str(e)[:300]}")
        continue
    try:
        t = gguf_throughput(llm)
        print(f"  {t['tok_per_s_mean']:.1f} +/- {t['tok_per_s_std']:.1f} tok/s")
        cpp_rows.append({
            "model":          entry["id"],
            "family":         entry["family"],
            "params_b":       entry["params_b"],
            "engine":         "llama.cpp",
            "tok_per_s_mean": t["tok_per_s_mean"],
            "tok_per_s_std":  t["tok_per_s_std"],
            "load_time_s":    t_load,
        })
        pd.DataFrame(cpp_rows).to_csv(OUT_DIR / "throughput_results_llamacpp.csv", index=False)
    except Exception as e:
        print(f"  [benchmark failed] {type(e).__name__}: {e}")
    del llm
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nSaved {OUT_DIR / 'throughput_results_llamacpp.csv'}")


---
# 7. Results 📊

Views, each reading whichever CSVs exist in `/kaggle/working/` (graceful if some are missing):

1. **HF accuracy table** (needs `accuracy_per_question.csv`, the run already on disk).
2. **llama.cpp accuracy + HF-vs-llama.cpp comparison** (needs `accuracy_per_question_llamacpp.csv` from §5). The `delta_pp` column answers the key question: does Qwen lose accuracy under Q4_K_M? Based on the §5 results (86.34% under llama.cpp), the answer is no.
3. **Throughput comparison** (needs both throughput CSVs from §6, plus the vLLM CSV if present) with a `speedup = cpp / hf` column.
4. **Bar chart**, saved to `throughput_comparison.png` for the report.

> Tables are colored with `pandas.Styler` so the winners are visible at a glance (green = better).


In [ ]:
# ---- View 1 + 1b: HF accuracy, llama.cpp accuracy, and side-by-side ----
import pandas as pd

OUT_DIR = Path("/kaggle/working")

def _style_acc(df, acc_cols):
    """Color the accuracy columns (green = high). Fallback: bare df."""
    try:
        sty = df.style.background_gradient(cmap="RdYlGn", subset=acc_cols, vmin=0, vmax=100)
        return sty.format({c: "{:.2f}" for c in acc_cols})
    except Exception:
        return df

# ---- View 1: HF accuracy ----
acc_csv = OUT_DIR / "accuracy_per_question.csv"
if acc_csv.exists():
    df_acc = pd.read_csv(acc_csv)
    print("=" * 70); print("HF accuracy"); print("=" * 70)
    by_model = (df_acc.groupby("model")
                      .agg(n=("correct", "size"),
                           accuracy_pct=("correct", lambda x: round(x.mean()*100, 2)),
                           mean_t_total=("t_total", lambda x: round(x.mean(), 2)),
                           live_tok_per_s=("tok_per_s_cot",
                                           lambda x: round(x[x > 0].mean() if (x > 0).any() else 0, 2)),
                           early_exit_pct=("early_exit", lambda x: round(x.mean()*100, 2)))
                      .reset_index())
    display(_style_acc(by_model, ["accuracy_pct"]))

    per_topic = (df_acc.groupby(["model", "topic"])["correct"]
                       .mean().mul(100).round(1).unstack("topic"))
    print("\nAccuracy per topic (%):")
    try:
        display(per_topic.style.background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=100))
    except Exception:
        display(per_topic)
else:
    print(f"[skip] {acc_csv.name} not found (HF accuracy run)")

# ---- View 1b: llama.cpp accuracy + comparison ----
def _acc_summary(df):
    return (df.groupby("model")
              .agg(n=("correct", "size"),
                   accuracy_pct=("correct", lambda x: round(x.mean()*100, 2)),
                   mean_t_total=("t_total", lambda x: round(x.mean(), 2)),
                   early_exit_pct=("early_exit", lambda x: round(x.mean()*100, 2)))
              .reset_index())

cpp_acc_csv = OUT_DIR / "accuracy_per_question_llamacpp.csv"
if cpp_acc_csv.exists():
    df_acc_cpp = pd.read_csv(cpp_acc_csv)
    print("\n" + "=" * 70); print("llama.cpp accuracy (Q4_K_M)"); print("=" * 70)
    cpp_summary = _acc_summary(df_acc_cpp)
    display(_style_acc(cpp_summary, ["accuracy_pct"]))

    # Sanity check: accuracy below chance => suspect pipeline (the §5 pre-fix bug)
    if (cpp_summary["accuracy_pct"] < 25).any():
        print("WARNING: accuracy below chance (25%). Likely a logits-scorer bug "
              "(see §5 note). Run debug_logits_once().")
    if "logits_found" in df_acc_cpp.columns:
        zero_found = (df_acc_cpp["logits_found"].fillna(0) == 0).mean() * 100
        print(f"Rows with 0 digits found in top_logprobs: {zero_found:.1f}% "
              "(high => scorer can't read logprobs).")

    # Per-topic breakdown for llama.cpp, same view as HF
    per_topic_cpp = (df_acc_cpp.groupby(["model", "topic"])["correct"]
                              .mean().mul(100).round(1).unstack("topic"))
    print("\nAccuracy per topic (%) — llama.cpp:")
    try:
        display(per_topic_cpp.style.background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=100))
    except Exception:
        display(per_topic_cpp)

    # Side-by-side: does Qwen keep its accuracy under Q4_K_M / llama.cpp?
    if acc_csv.exists():
        hf_sum  = _acc_summary(pd.read_csv(acc_csv))[["model", "accuracy_pct"]]
        hf_sum  = hf_sum.rename(columns={"accuracy_pct": "hf_acc_pct"})
        cpp_sum = cpp_summary[["model", "accuracy_pct"]].rename(
            columns={"accuracy_pct": "cpp_acc_pct"})
        acc_cmp = hf_sum.merge(cpp_sum, on="model", how="inner")
        if not acc_cmp.empty:
            acc_cmp["delta_pp"] = (acc_cmp["cpp_acc_pct"] - acc_cmp["hf_acc_pct"]).round(2)
            print("\nAccuracy: HF (bnb 4-bit)  vs  llama.cpp (Q4_K_M)")
            print("delta_pp > 0  => llama.cpp MORE accurate; < 0 => quality lost")
            try:
                display(acc_cmp.style.background_gradient(
                    cmap="RdYlGn", subset=["delta_pp"], vmin=-10, vmax=10))
            except Exception:
                display(acc_cmp)
else:
    print(f"\n[skip] {cpp_acc_csv.name} not found (run §5 first)")


In [ ]:
# ---- View 2: throughput HF vs llama.cpp (+ vLLM if present) ----
hf_csv   = OUT_DIR / "throughput_results.csv"
cpp_csv  = OUT_DIR / "throughput_results_llamacpp.csv"
vllm_csv = OUT_DIR / "throughput_results_vllm.csv"   # from §8 (single-stream)

cmp = None
if hf_csv.exists() and cpp_csv.exists():
    df_hf  = pd.read_csv(hf_csv)
    df_cpp = pd.read_csv(cpp_csv)
    hf_s   = df_hf[["model", "tok_per_s_mean", "tok_per_s_std"]].rename(
        columns={"tok_per_s_mean": "hf_tok_s", "tok_per_s_std": "hf_std"})
    cpp_s  = df_cpp[["model", "tok_per_s_mean", "tok_per_s_std"]].rename(
        columns={"tok_per_s_mean": "cpp_tok_s", "tok_per_s_std": "cpp_std"})
    cmp = hf_s.merge(cpp_s, on="model", how="outer")
    cmp["speedup"] = (cmp["cpp_tok_s"] / cmp["hf_tok_s"]).round(2)

    # Add the vLLM single-stream column if the CSV exists
    if vllm_csv.exists():
        df_v = pd.read_csv(vllm_csv)
        v_s = df_v[["model", "tok_per_s_mean", "tok_per_s_std"]].rename(
            columns={"tok_per_s_mean": "vllm_tok_s", "tok_per_s_std": "vllm_std"})
        cmp = cmp.merge(v_s, on="model", how="outer")
        cmp["vllm_speedup"] = (cmp["vllm_tok_s"] / cmp["hf_tok_s"]).round(2)

    cmp = cmp.round(2)
    print("=" * 70)
    print("Throughput: HF generate()+bnb int4  vs  llama.cpp+GGUF Q4_K_M"
          + ("  vs  vLLM" if vllm_csv.exists() else ""))
    print("=" * 70)
    try:
        spd = [c for c in ["speedup", "vllm_speedup"] if c in cmp.columns]
        display(cmp.style.background_gradient(cmap="Greens", subset=spd))
    except Exception:
        display(cmp)
    cmp.to_csv(OUT_DIR / "throughput_comparison.csv", index=False)
elif hf_csv.exists():
    print("=" * 70); print("HF throughput only (run §6 RUN C to add the llama.cpp column)"); print("=" * 70)
    display(pd.read_csv(hf_csv).round(2))
elif cpp_csv.exists():
    print("=" * 70); print("llama.cpp throughput only (run §6 RUN A to add the HF column)"); print("=" * 70)
    display(pd.read_csv(cpp_csv).round(2))
else:
    print("[skip] no throughput CSVs found (run §6)")


In [ ]:
# ---- View 3: bar chart (engine comparison) ----
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc", "axes.grid": True,
    "grid.color": "#e6e6e6", "font.size": 11,
})

if cmp is not None:
    has_vllm = ("vllm_tok_s" in cmp.columns) and cmp["vllm_tok_s"].notna().any()
    plot_df = cmp.dropna(subset=["hf_tok_s", "cpp_tok_s"]).copy()
    if not plot_df.empty:
        labels = plot_df["model"].str.split("/").str[-1].tolist()
        x = np.arange(len(labels))

        # Series to plot: (column, std column, label, color)
        series = [
            ("hf_tok_s",  "hf_std",  "HF generate() + bnb int4", "#4C72B0"),
            ("cpp_tok_s", "cpp_std", "llama.cpp + GGUF Q4_K_M",  "#DD8452"),
        ]
        if has_vllm:
            series.append(("vllm_tok_s", "vllm_std", "vLLM (single-stream)", "#55A868"))

        n = len(series)
        w = 0.8 / n
        # Centered offsets: n=2 -> [-0.5w, +0.5w]; n=3 -> [-w, 0, +w]
        offsets = [(i - (n - 1) / 2) * w for i in range(n)]

        fig, ax = plt.subplots(figsize=(12, 6))
        drawn = []
        for (col, std_col, lab, color), off in zip(series, offsets):
            yerr = plot_df[std_col] if std_col in plot_df.columns else None
            bset = ax.bar(x + off, plot_df[col], w, yerr=yerr, capsize=4,
                          label=lab, color=color, edgecolor="black", linewidth=.6)
            ax.bar_label(bset, fmt="%.0f", padding=2, fontsize=9)
            drawn.append(plot_df[col].max())

        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right")
        ax.set_ylabel("Decoding throughput (tokens/sec)")
        ax.set_title("Engine comparison on T4 (single GPU, 4-bit weights)",
                     fontsize=14, fontweight="bold")
        ax.legend(frameon=True)
        ax.grid(axis="y", alpha=0.4); ax.set_axisbelow(True)

        # Speedup annotations (llama.cpp vs HF) above each group
        top = max(drawn)
        for xi, sp in zip(x, plot_df["speedup"]):
            ax.text(xi, top * 1.10, f"{sp:.2f}x", ha="center",
                    fontsize=11, fontweight="bold", color="#DD8452")
        ax.set_ylim(0, top * 1.22)

        plt.tight_layout()
        plt.savefig(OUT_DIR / "throughput_comparison.png", dpi=140, bbox_inches="tight")
        plt.show()
        print(f"Saved {OUT_DIR / 'throughput_comparison.png'}")


## Global Helper Functions
These utility functions handle answer extraction from raw LLM outputs and format options dictionaries. They are used across both the Maths and News pipelines.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re, gc, time, statistics, csv, os, json, threading, traceback
import pandas as pd
import requests

# ── Answer extraction helpers ──
def extract_answer(answer_text):
    if not answer_text: return 0
    m = re.search(r'"answer"\s*:\s*"?([0-3])"?', answer_text)
    if m: return int(m.group(1))
    m = re.search(r"\b([0-3])\b", answer_text)
    if m: return int(m.group(1))
    return 0

def extract_confidence(answer_text):
    m = re.search(r'"confidence"\s*:\s*([0-9\.]+)', answer_text)
    if m:
        try: return max(0.0, min(1.0, float(m.group(1))))
        except: pass
    return 0.5

def extract_json_object(text):
    m = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except: return None

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

def get_options_dict(question):
    return {int(opt.id): str(opt.text) for opt in question.options}

print('Helpers loaded.')

# 8. vLLM — Why It Was Excluded From the Benchmark 

vLLM is a GPU inference library designed to serve many requests in parallel, 
built on two central ideas — **PagedAttention** (KV cache managed in 
fixed-size pages, like an OS's virtual memory) and **continuous batching** 
(finished sequences are swapped out at every decoding step, with no need to 
wait for the full batch). Under concurrent load it achieves aggregate 
throughput several times higher than HuggingFace `generate()`.

We attempted to include vLLM as a third engine, but installation on Kaggle 
failed due to **ABI incompatibility with the image's torch stack**: the 
prebuilt vLLM wheels were compiled against a torch version different from 
the one shipped in the Kaggle environment, producing unresolved C++ symbols 
and missing imports at load time. Fixing this would have required 
downgrading torch (to 2.4), which would have in turn broken the 
already-working pipeline. The only clean path would have been a **separate Kaggle session dedicated to vLLM**, with its own environment.

Setting aside installation issues, there are two independent reasons why 
vLLM would have offered limited value in our setup:

1. **Single-stream workload.** Our Millionaire pipeline answers one 
   question at a time. vLLM's headline advantage (continuous batching) 
   only materializes under concurrent requests.
2. **T4 GPU (Turing architecture, sm_75).** vLLM's kernels are optimized 
   for Ampere and later; on Turing several features require fallbacks, further        reducing the expected advantage. 


**Conclusion.** vLLM remains the reference engine for multi-user or 
offline batch scenarios over many requests, and the notebook ships with 
the infrastructure (`load_vllm`, `vllm_throughput_single`, 
`vllm_throughput_batched`) to measure it in a dedicated session. For the 
single-stream benchmark presented here, **llama.cpp + GGUF Q4_K_M** 
remains the right choice: it triples/quadruples HF throughput on a single 
T4, runs from a single install, and requires no changes to the stack.

---
# 9. Maths 🧮

When dealing with the Maths section, our first idea was to leverage tools like LangChain to provide the LLM with a calculator. The assumption was that it wouldn't get the calculations wrong if it could just use an external calculator. 

In the initial design phase, we considered using the **LangChain** framework to provide the chatbot with a set of dedicated *tools*. The goal was to delegate mathematical calculations to external functions to ensure deterministic and precise results. 

However, we almost immediately encountered some structural limitations with this approach:
* **Lack of versatility:** The tool-based system proved to be excessively rigid.
* **Domain unpredictability:** To make the agent truly effective, we would have needed to know *a priori* and pre-program all possible types of calculations and formulas required by the game's questions (ranging from arithmetic to complex algebra), which is clearly an unfeasible task.

The collected data confirmed these limitations: by providing the model with a small set of basic mathematical tools, we noticed that **the accuracy in experimental runs on math questions struggled to exceed 60%**. The model simply failed because it did not have the specific tools available to solve the majority of the more advanced questions.

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
import math
import sympy as sp

# 1. Definition of a rigid set of Math Tools
@tool
def add(a: float, b: float) -> float:
    """Adds two numbers. Use this tool for addition."""
    return a + b

@tool
def calculate_logarithm(x: float, base: float = math.e) -> float:
    """Calculates the logarithm of x to a given base (default is natural log)."""
    return math.log(x, base)

@tool
def solve_linear_system(eq1_coeffs: list, eq2_coeffs: list, constants: list) -> dict:
    """
    Solves a system of two linear equations.
    eq1_coeffs: [x, y] coefficients of the first equation
    eq2_coeffs: [x, y] coefficients of the second equation
    constants: constant terms [c1, c2]
    """
    x, y = sp.symbols('x y')
    eq1 = sp.Eq(eq1_coeffs[0]*x + eq1_coeffs[1]*y, constants[0])
    eq2 = sp.Eq(eq2_coeffs[0]*x + eq2_coeffs[1]*y, constants[1])
    return sp.solve((eq1, eq2), (x, y))

@tool
def compound_interest(principal: float, rate: float, years: float) -> float:
    """
    Computes the result of applying a specific interest rate to a principal for a number of years.
    principal: initial amount
    rate: annual interest rate as a percentage (e.g., 5 for 5%)
    years: number of years
    """
    return principal * ((1 + rate / 100) ** years)

@tool
def exponentiate(base: float, power: float) -> float:
    """
    Computes base^power, i.e., raises the base to the specified power.
    """
    return base ** power

@tool
def calculate_factorial(n: int) -> int:
    """
    Calculates the factorial of a non-negative integer n (n!).
    Use this for combinatorics and probability problems.
    """
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    return math.factorial(n)

# Group the tools together
math_tools = [
    add, 
    calculate_logarithm, 
    solve_linear_system, 
    compound_interest, 
    exponentiate, 
    calculate_factorial
]

# 2. LLM Initialization (must support Tool Calling)
llm = ChatOpenAI(model="qwen-2.5-7b", temperature=0) # Conceptual example

# 3. Prompt and Agent Creation
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a math assistant. Use the tools at your disposal to solve problems step-by-step."),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, math_tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=math_tools, verbose=True)

To solve this, we decided to give the model Program-Aided Language (PAL) capabilities. We engineered the prompt constraints to ensure the model outputs a short, efficient Python script to solve the problem (using libraries like `math`, `statistics`, or `sympy`). We developed a pipeline of **generate -> execute -> choose**, where we extract the Python code from the response, run it in a safe environment, and then map the execution output to one of the multiple-choice options.

To make the pipeline robust, we engineered it by logging the answers and time taken for each part and analyzing bottlenecks. For example, we implemented timeout mechanisms on code generation, tracked early exits, and monitored execution failures to refine our system prompts and constraints.

Later, we found a dataset that we could use for RAG. We implemented a BM25-based RAG pipeline to retrieve relevant mathematical formulas and concepts, which are then injected into the prompt along with the question before generating the PAL script.

**Results on Maths:**
- Vanilla Qwen3.5-9B: ~61%
- Qwen3.5-9B + PAL: 65-66%
- Qwen3.5-9B + RAG + PAL: 84-85%

Below we demonstrate the code for both the PAL-only and the RAG+PAL implementations.


## 9.1 PAL-only Pipeline
First, we define the secure Python execution environment and run the PAL-only benchmark. **~67% accuracy**

In [ ]:
import concurrent.futures
import sys
import io
import sympy as sp
import math, statistics, re, json

def _execute_python_code(code_str):
    old_stdout = sys.stdout
    redirected_output = io.StringIO()
    sys.stdout = redirected_output
    
    safe_globals = {
        "sp": sp, "sympy": sp,
        "math": math, "statistics": statistics,
        "print": print
    }
    
    try:
        exec(code_str, safe_globals)
        output = redirected_output.getvalue().strip()
        if not output:
            for var in ['ans', 'answer', 'result', 'sol', 'final_result', 'x', 'y', 'z']:
                if var in safe_globals:
                    output = str(safe_globals[var])
                    break
            if not output:
                output = "No printed output."
    except Exception as e:
        output = f"Execution Error: {type(e).__name__}: {e}"
    finally:
        sys.stdout = old_stdout
        
    return str(output)[:500]  # truncate to prevent massive outputs

def execute_python_code(code_str, timeout=4.0):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(_execute_python_code, code_str)
        try:
            return future.result(timeout=timeout)
        except concurrent.futures.TimeoutError:
            return "Timeout Error: Script took too long."
        except Exception as e:
            return f"Error: {e}"

def tool_option_checker(question, output_str):
    try:
        float_val = float(output_str)
    except ValueError:
        float_val = None
    
    if float_val is not None:
        match_id = match_value_to_option(float_val, question, tolerance=1e-3)
        if match_id is not None:
            return {
                "answer_id": match_id,
                "confidence": 0.95,
                "text": str(output_str),
                "reason": f"Matched printed value {output_str}"
            }
    return {
        "answer_id": 0,
        "confidence": 0.0,
        "text": "",
        "reason": "No matches"
    }
print("PAL Executor loaded.")


In [ ]:
# --- PAL Pipeline Orchestrator ---
PYTHON_GENERATOR_SYSTEM_PROMPT = """You are an expert Python programmer solving math multiple-choice problems.
Write a short, efficient Python script using `math`, `statistics`, or `sympy` (imported as `sp`) to solve the problem. DO NOT output any conversational text or markdown before or after the code.
You MUST assign your final answer to a variable named 'ans' AND print(ans) at the end. Do not print intermediate steps.

CRITICAL RULES:
1. If using `sp.solve()`, it returns a list. Check for emptiness: `ans = sol[0] if sol else None`. Do not pass lists to `sp.Eq()`.
2. ABSTRACT ALGEBRA: For Modular Arithmetic or Polynomial Rings (e.g. Z_n), DO NOT use SymPy ring generators. Use basic Python integers and apply modulo `% n` to the final result.
3. INEQUALITIES: SymPy solve() fails on inequalities (e.g. |x| < 5). If asking for integers, write a basic Python `for` loop to check integers in a reasonable range.
4. STATISTICS & PROBABILITY: Prefer `statistics` module and `math.comb()`. For word problems: "A and B" = joint P(A ∩ B); "A given B" = conditional P(A|B); "no A" = 1 - P(A). Do NOT multiply probabilities unless independence is stated. Use P(A ∩ B) = P(A) - P(A ∩ ¬B) when applicable.
5. THEORETICAL QUESTIONS: If the problem requires NO calculations and is purely conceptual, do NOT write a script. Output exactly the word `SKIP` and nothing else.
6. ZERO COMMENTS ALLOWED: Do NOT write ANY python comments (`#`) in your code. Writing comments causes a FATAL TIMEOUT. Output ONLY the raw code.
7. INFINITE SERIES: Prefer symbolic closed forms (e.g. sum x^k = 1/(1-x), sum kx^k = x/(1-x)^2) over numerical loops. Avoid while loops for infinite sums.

Example:
Problem: If 2x + 5 = 15, what is x?
Options:
0: 5
1: 10
2: 2

```python
import sympy as sp
x = sp.symbols('x')
eq = sp.Eq(2*x + 5, 15)
sol = sp.solve(eq, x)
ans = sol[0] if sol else None
print(ans)
```
"""

FAST_SOLVER_SYSTEM_PROMPT = """You are a fast, highly accurate math multiple-choice solver.
Use the output of the executed Python script to select the correct option.
WARNING: The script output might be an error or a partial step. Use your reasoning to select the final option. Do not show your work. Return ONLY JSON.
CRITICAL RULE: Never output the computed numeric value as the final answer ID! Always map the computed value to the matching option index. If the computed value is 2, and Option 1 is "2", you MUST return answer 1, not 2.
If no option exactly matches the script output, do not blindly guess the nearest option unless confidence is low.

Example JSON output:
{"answer": 2, "confidence": 0.95}
"""

def llm_generate_python(question, tokenizer, model, max_time=26.0):
    messages = [
        {"role": "system", "content": PYTHON_GENERATOR_SYSTEM_PROMPT},
        {"role": "user", "content": f"Problem:\n{question.text}\n\nWrite Python code:"}
    ]
    try: inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt", enable_thinking=False).to(model.device)
    except: inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
        
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=400, do_sample=False, max_time=max_time, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    
    if "SKIP" in text[:30]:
        return "SKIP", text
        
    m = re.search(r"```python\n(.*?)(?:```|$)", text, re.DOTALL)
    if m: return m.group(1).strip(), text
    
    if "=" in text or "import " in text or "print(" in text:
        return text.replace("```python", "").replace("```", "").strip(), text
        
    return "", text

def build_pipeline_prompt(question, script_output, option_check):
    options = get_options_dict(question)
    prompt = f"Solve this math multiple-choice problem quickly and accurately.\n\nProblem:\n{question.text}\n\nOptions:\n"
    for opt_id, opt_text in options.items(): prompt += f"{opt_id}: {opt_text}\n"
    
    if "Error:" in script_output or "No printed output" in script_output or script_output.strip() == "":
        pass # Hide syntax errors from Qwen to prevent confusion
    elif script_output == "SKIP":
        pass # It's a theoretical question, no tool output
    else:
        prompt += f"\nPython Script Execution Output:\n{script_output}\n"
        if option_check["confidence"] > 0:
            prompt += f"\nNote: The script output perfectly matches Option {option_check['answer_id']}. Verify if this is the final answer or just a partial step!\n"
    prompt += "\nReturn only JSON:\n{\"answer\": 0, \"confidence\": 0.0}\n"
    return prompt.strip()

def choose_math_answer_with_tools(question, tokenizer, model):
    start = time.perf_counter()
    
    # 1. LLM Python Code Generation
    code_str, raw_code_text = llm_generate_python(question, tokenizer, model, max_time=26.0)
    
    # Free KV cache from first pass
    torch.cuda.empty_cache()
    
    # 2. Execute Python Code
    elapsed_after_gen = time.perf_counter() - start
    
    if code_str == "SKIP":
        script_output = "SKIP"
    elif elapsed_after_gen > 23.0:
        script_output = "Timeout: Code generation took too long. Proceeding without tools."
    elif code_str:
        # We only give it whatever time is left before 25s, max 4s
        exec_timeout = max(0.5, min(4.0, 25.0 - elapsed_after_gen))
        script_output = execute_python_code(code_str, timeout=exec_timeout)
    else:
        script_output = "No valid Python code generated."
        
    # 3. Option Checker
    if script_output in ["SKIP", "Timeout: Code generation took too long. Proceeding without tools.", "No valid Python code generated."]:
        option_check = {"answer_id": 0, "confidence": 0.0, "text": "", "reason": "No matches"}
    else:
        option_check = tool_option_checker(question, script_output)
        
    if option_check["confidence"] >= 0.9:
        torch.cuda.empty_cache()
        return {
            "generated_code": code_str,
            "script_output": script_output,
            "raw_code_text": raw_code_text,
            "answer_id": option_check["answer_id"],
            "confidence": option_check["confidence"],
            "source": "pal_early_exit",
            "raw_output": "Early Exit matched Option.",
            "elapsed": time.perf_counter() - start,
            "tools_used": ["python_executor"]
        }
    # 4. LLM Answer Generator
    elapsed = time.perf_counter() - start
    remaining = max(3.0, min(12.0, 28.0 - elapsed))
    
    messages = [
        {"role": "system", "content": FAST_SOLVER_SYSTEM_PROMPT},
        {"role": "user", "content": build_pipeline_prompt(question, script_output, option_check)},
    ]
    try: inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt", enable_thinking=False).to(model.device)
    except: inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
        
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False, max_time=remaining, pad_token_id=tokenizer.eos_token_id)
        
    text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    
    ans_id = extract_answer(text)
    conf = extract_confidence(text)
    
    if conf < 0.1 and option_check["confidence"] > 0:
        ans_id = option_check["answer_id"]
        conf = 0.5
        text = "LLM failed format, fallback to option check. " + text
        
    torch.cuda.empty_cache()
    return {
        "generated_code": code_str,
        "script_output": script_output,
        "raw_code_text": raw_code_text,
        "answer_id": ans_id,
        "confidence": conf,
        "source": "pal_with_qwen",
        "raw_output": text,
        "elapsed": time.perf_counter() - start,
        "tools_used": ["python_executor"] if code_str and code_str != "SKIP" else []
    }
print("PAL Orchestrator loaded.")


In [ ]:
models_to_test = ["Qwen/Qwen3.5-9B"]

NUM_ITERATIONS_PER_TOPIC = 30
DELAY_BETWEEN_CALLS = 2.0
MAX_TOTAL_SECONDS = 30.0

benchmark_results = []


In [ ]:
try:
    competitions = client.competitions.list_all()
    print("\nFound topics to benchmark:")
    for c in competitions:
        print(f" - {c.name} (ID: {c.id})")
except Exception as e:
    print(f"Could not fetch competitions: {e}")
    competitions = []

for model_id in models_to_test:
    print(f"\n{'=' * 70}\nTesting model: {model_id}\n{'=' * 70}")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        from transformers import BitsAndBytesConfig
        quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto", quantization_config=quantization_config, trust_remote_code=True)
        model.eval()
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id
    except Exception as e:
        print(f"Failed to load model {model_id}: {e}\nSkipping...\n")
        continue

    model_overall_correct = 0
    model_overall_wrong = 0
    model_overall_timeouts = 0
    model_overall_earnings = 0
    question_logs = []
    model_answer_times = []
    topic_stats = {}

    for comp in competitions:
        if comp.id != 3: continue
        print(f"\n  >> Starting Topic: {comp.name} (ID: {comp.id})")
        comp_correct = 0
        comp_wrong = 0
        comp_timeouts = 0
        comp_answer_times = []

        for i in range(NUM_ITERATIONS_PER_TOPIC):
            print(f"     Game {i + 1}/{NUM_ITERATIONS_PER_TOPIC}... ", end="")
            time.sleep(DELAY_BETWEEN_CALLS)
            try:
                game = client.game.start(competition_id=comp.id)
            except Exception as e:
                print(f'Start error: {e}, retrying...')
                time.sleep(3)
                continue

            while game.in_progress:
                question = game.current_question
                if not question: break
                question_received_at = time.perf_counter()

                decision = choose_math_answer_with_tools(question, tokenizer, model)

                answer_id = decision["answer_id"]
                submitted_at = time.perf_counter()
                answer_time_seconds = submitted_at - question_received_at

                print(
                    f"answer={answer_id} "
                    f"source={decision.get('source')} "
                    f"tools={decision.get('tools_used', [])} "
                    f"conf={decision.get('confidence', 0.0):.2f} "
                    f"time={answer_time_seconds:.2f}s"
                )

                comp_answer_times.append(answer_time_seconds)
                model_answer_times.append(answer_time_seconds)
                time.sleep(DELAY_BETWEEN_CALLS)

                try:
                    result = game.answer(answer_id)
                except Exception as e:
                    print(f'Answer error: {e}, skipping question...')
                    time.sleep(3)
                    break
                options_dict = get_options_dict(question)
                question_logs.append({
                    "Topic": comp.name,
                    "Question": question.text,
                    "Options": json.dumps(options_dict),
                    "Generated Code": decision.get("generated_code", ""),
                    "Script Output": decision.get("script_output", ""),
                    "Raw Final LLM Output": decision.get("raw_output", ""),
                    "Our Answer ID": answer_id,
                    "Our Answer Text": options_dict.get(answer_id, ""),
                    "Correct": result.correct,
                    "Time Taken (s)": answer_time_seconds,
                    "API Raw Response": str(result)
                })
                if result.correct:
                    comp_correct += 1
                    if result.game_over: model_overall_earnings += result.earned_amount
                elif result.timed_out:
                    comp_timeouts += 1
                    model_overall_earnings += result.earned_amount
                else:
                    comp_wrong += 1
                    model_overall_earnings += result.earned_amount
            print("Done.")

        topic_stats[comp.name] = {
            "correct": comp_correct,
            "wrong": comp_wrong,
            "timeouts": comp_timeouts,
            "answer_times": comp_answer_times,
        }
        model_overall_correct += comp_correct
        model_overall_wrong += comp_wrong
        model_overall_timeouts += comp_timeouts

    overall_total = model_overall_correct + model_overall_wrong + model_overall_timeouts
    overall_acc = (model_overall_correct / overall_total * 100) if overall_total > 0 else 0

    flat_result = {
        "Model": model_id,
        "Overall Accuracy (%)": round(overall_acc, 2),
        "Total Earnings ($)": model_overall_earnings,
        "Avg Time / Question (s)": round(statistics.mean(model_answer_times), 3) if model_answer_times else None,
    }

    for t_name, t_stats in topic_stats.items():
        t_total = t_stats["correct"] + t_stats["wrong"] + t_stats["timeouts"]
        t_acc = (t_stats["correct"] / t_total * 100) if t_total > 0 else 0
        times = t_stats["answer_times"]
        flat_result[f"{t_name} Accuracy (%)"] = round(t_acc, 2)
        flat_result[f"{t_name} Avg Time / Question (s)"] = round(statistics.mean(times), 3) if times else None

    benchmark_results.append(flat_result)
    pd.DataFrame(question_logs).to_csv(f'/kaggle/working/question_logs_{model_id.replace("/", "_")}.csv', index=False)
    print(f"\nSaved detailed question logs for {model_id}!")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

print("\nBenchmarking Complete!")


In [ ]:
import pandas as pd

df_results = pd.DataFrame(benchmark_results)
if not df_results.empty:
    display(df_results)
    output_csv = "/kaggle/working/benchmark_results.csv"
    df_results.to_csv(output_csv, index=False)
    print(f"\nSaved final results to: {output_csv}")

# Optional local evaluation on CSV (if present)
csv_path = "/kaggle/input/math_samples.csv"
if os.path.exists(csv_path):
    print(f"\nFound local math samples at {csv_path}, running batch evaluation...")
    df = pd.read_csv(csv_path, sep=None, engine='python')
    # This block can be expanded to run tools on the DF
    print("Batch evaluation logic goes here.")
else:
    print("\nNo local CSV samples found. Skipping batch evaluation.")


## 9.2 RAG + PAL Pipeline
Next, we introduce the RAG database to retrieve relevant formulas before generating the PAL script. 
**~85% accuracy**

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from datasets import load_dataset
import time

print("==================================================")
print("INITIALIZING RAG DATABASE")
print("==================================================")
t0 = time.perf_counter()
print("Loading AI-MO/NuminaMath-CoT dataset...")
ds = load_dataset("AI-MO/NuminaMath-CoT", split="train")

rag_problems = [item['problem'] for item in ds]
rag_solutions = [item['solution'] for item in ds]

print(f"Building TF-IDF Index for {len(rag_problems)} rows...")
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(rag_problems)
t1 = time.perf_counter()
print(f"RAG Index built successfully in {t1-t0:.2f} seconds!")

def retrieve_rag_context(query, top_k=1):
    t_start = time.perf_counter()
    query_vec = vectorizer.transform([query])
    cosine_similarities = linear_kernel(query_vec, tfidf_matrix).flatten()
    top_indices = cosine_similarities.argsort()[-top_k:][::-1]
    
    contexts = []
    matched_probs = []
    for i, idx in enumerate(top_indices):
        score = cosine_similarities[idx]
        # Only include if there is some meaningful overlap
        if score > 0.05:
            contexts.append(f"--- Example {i+1} ---\nSimilar Problem:\n{rag_problems[idx]}\n\nStep-by-step Solution:\n{rag_solutions[idx]}")
            matched_probs.append(f"[{score:.3f}] {rag_problems[idx]}")
            snippet = rag_problems[idx][:80].replace('\n', ' ')
            print(f"  [RAG] Matched problem (Score: {score:.3f}): '{snippet}...'")
        else:
            print(f"  [RAG] Skipped match (Score too low: {score:.3f})")
            
    t_end = time.perf_counter()
    print(f"  [RAG] Retrieval took {t_end - t_start:.3f}s")
    
    if not contexts:
        return "", []
        
    return "\n\n---\n\n".join(contexts), matched_probs


In [ ]:
# --- PAL Pipeline Orchestrator ---
PYTHON_GENERATOR_SYSTEM_PROMPT = """You are an expert Python programmer solving math multiple-choice problems.
Write a short, efficient Python script using `math`, `statistics`, or `sympy` (imported as `sp`) to solve the problem. DO NOT output any conversational text or markdown before or after the code.
You MUST assign your final answer to a variable named 'ans' AND print(ans) at the end. Do not print intermediate steps.

CRITICAL RULES:
1. If using `sp.solve()`, it returns a list. Check for emptiness: `ans = sol[0] if sol else None`. Do not pass lists to `sp.Eq()`.
2. ABSTRACT ALGEBRA: For Modular Arithmetic or Polynomial Rings (e.g. Z_n), DO NOT use SymPy ring generators. Use basic Python integers and apply modulo `% n` to the final result.
3. INEQUALITIES: SymPy solve() fails on inequalities (e.g. |x| < 5). If asking for integers, write a basic Python `for` loop to check integers in a reasonable range.
4. STATISTICS & PROBABILITY: Prefer `statistics` module and `math.comb()`. For word problems: "A and B" = joint P(A ∩ B); "A given B" = conditional P(A|B); "no A" = 1 - P(A). Do NOT multiply probabilities unless independence is stated. Use P(A ∩ B) = P(A) - P(A ∩ ¬B) when applicable.
5. THEORETICAL QUESTIONS: If the problem requires NO calculations and is purely conceptual, do NOT write a script. Output exactly the word `SKIP` and nothing else.
6. ZERO COMMENTS ALLOWED: Do NOT write ANY python comments (`#`) in your code. Writing comments causes a FATAL TIMEOUT. Output ONLY the raw code.
7. INFINITE SERIES: Prefer symbolic closed forms (e.g. sum x^k = 1/(1-x), sum kx^k = x/(1-x)^2) over numerical loops. Avoid while loops for infinite sums.

Example:
Problem: If 2x + 5 = 15, what is x?
Options:
0: 5
1: 10
2: 2

```python
import sympy as sp
x = sp.symbols('x')
eq = sp.Eq(2*x + 5, 15)
sol = sp.solve(eq, x)
ans = sol[0] if sol else None
print(ans)
```
"""

FAST_SOLVER_SYSTEM_PROMPT = """You are a fast, highly accurate math multiple-choice solver.
Use the output of the executed Python script to select the correct option.
WARNING: The script output might be an error or a partial step. Use your reasoning to select the final option. Do not show your work. Return ONLY JSON.
CRITICAL RULE: Never output the computed numeric value as the final answer ID! Always map the computed value to the matching option index. If the computed value is 2, and Option 1 is "2", you MUST return answer 1, not 2.
If no option exactly matches the script output, do not blindly guess the nearest option unless confidence is low.

Example JSON output:
{"answer": 2, "confidence": 0.95}
"""

def llm_generate_python(question, llm, rag_hint=""): 
    user_msg = f"Problem:\n{question.text}{rag_hint}\n\nCRITICAL: Output ONLY valid Python code. Do not write any explanations. Write Python code:"
    prompt = f"<|im_start|>system\n{PYTHON_GENERATOR_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n```python\n"
    print("  [LLM] Generating Python Code...")
    t_llm0 = time.perf_counter()
    try:
        outputs = llm.create_completion(prompt=prompt, max_tokens=700, temperature=0.0, stop=["```", "<|im_end|>"])
    except Exception as e:
        print("Fallback LLM error:", e)
        return "", "Error"
    t_llm1 = time.perf_counter()
    print(f"  [LLM] Code Generation took {t_llm1 - t_llm0:.2f}s")
    text = outputs["choices"][0]["text"] or ""
    text = text.strip()
    
    if "SKIP" in text[:30]:
        return "SKIP", text
    
    # Since we prefilled ```python\n, the output is JUST the code!
    return text, text

def build_pipeline_prompt(question, script_output, option_check, rag_hint=""):
    options = get_options_dict(question)
    prompt = f"Solve this math multiple-choice problem quickly and accurately.\n\nProblem:\n{question.text}{rag_hint}\n\nOptions:\n"
    for opt_id, opt_text in options.items(): prompt += f"{opt_id}: {opt_text}\n"
    
    if "Error:" in script_output or "No printed output" in script_output or script_output.strip() == "":
        pass # Hide syntax errors from Qwen to prevent confusion
    elif script_output == "SKIP":
        pass # It's a theoretical question, no tool output
    else:
        prompt += f"\nPython Script Execution Output:\n{script_output}\n"
        if option_check["confidence"] > 0:
            prompt += f"\nNote: The script output perfectly matches Option {option_check['answer_id']}. Verify if this is the final answer or just a partial step!\n"
    prompt += "\nReturn only JSON:\n{\"answer\": 0, \"confidence\": 0.0}\n"
    return prompt.strip()

def choose_math_answer_with_tools(question, llm):
    start = time.perf_counter()
    
    # --- RAG RETRIEVAL ---
    rag_context, matched_probs = retrieve_rag_context(question.text, top_k=3)
    rag_hint = f"\n\nHint: Here are some highly similar problems and their solutions:\n\n{rag_context}\n\nUse the mathematical logic from these examples to help solve the current problem." if rag_context else ""
    # ----------------------
    
    # 1. LLM Python Code Generation
    code_str, raw_code_text = llm_generate_python(question, llm, rag_hint=rag_hint)
    
    # 2. Execute Python Code
    elapsed_after_gen = time.perf_counter() - start
    
    if code_str == "SKIP":
        script_output = "SKIP"
    elif elapsed_after_gen > 24.0:
        script_output = "Timeout: Code generation took too long. Proceeding without tools."
    elif code_str:
        exec_timeout = max(0.5, min(6.0, 25.0 - elapsed_after_gen))
        print("  [EXEC] Running Python Script...")
        t_ex0 = time.perf_counter()
        script_output = execute_python_code(code_str, timeout=exec_timeout)
        t_ex1 = time.perf_counter()
        print(f"  [EXEC] Script execution took {t_ex1 - t_ex0:.2f}s")
    else:
        script_output = "No valid Python code generated."
        
    # 3. Option Checker
    if script_output in ["SKIP", "Timeout: Code generation took too long. Proceeding without tools.", "No valid Python code generated."]:
        option_check = {"answer_id": 0, "confidence": 0.0, "text": "", "reason": "No matches"}
    else:
        option_check = tool_option_checker(question, script_output)
        
    if option_check["confidence"] >= 0.9:
        return {
            "generated_code": code_str,
            "script_output": script_output,
            "raw_code_text": raw_code_text,
            "answer_id": option_check["answer_id"],
            "confidence": option_check["confidence"],
            "source": "pal_early_exit",
            "raw_output": "Early Exit matched Option.",
            "elapsed": time.perf_counter() - start,
            "tools_used": ["python_executor"],
            "rag_matches": " || ".join(matched_probs)
        }
    # 4. LLM Answer Generator
    elapsed = time.perf_counter() - start
    remaining = max(3.0, min(12.0, 27.0 - elapsed))
    max_toks = max(10, min(128, int(remaining * 30)))
    
    # We can also use create_completion here to force JSON output
    user_msg2 = build_pipeline_prompt(question, script_output, option_check)
    prompt2 = f"<|im_start|>system\n{FAST_SOLVER_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_msg2}<|im_end|>\n<|im_start|>assistant\n{{\n"
    print("  [LLM] Generating Answer...")
    t_llm0 = time.perf_counter()
    try:
        outputs = llm.create_completion(prompt=prompt2, max_tokens=max_toks, temperature=0.0, stop=["<|im_end|>"])
        text = "{" + (outputs["choices"][0]["text"] or "").strip()
    except Exception:
        text = "{}"
    t_llm1 = time.perf_counter()
    print(f"  [LLM] Answer Generation took {t_llm1 - t_llm0:.2f}s")
    
    ans_id = extract_answer(text)
    conf = extract_confidence(text)
    
    if conf < 0.1 and option_check["confidence"] > 0:
        ans_id = option_check["answer_id"]
        conf = 0.5
        text = "LLM failed format, fallback to option check. " + text
        
    return {
        "generated_code": code_str,
        "script_output": script_output,
        "raw_code_text": raw_code_text,
        "answer_id": ans_id,
        "confidence": conf,
        "source": "pal_with_qwen",
        "raw_output": text,
        "elapsed": time.perf_counter() - start,
        "tools_used": ["python_executor"] if code_str and code_str != "SKIP" else [],
        "rag_matches": " || ".join(matched_probs)
    }


In [ ]:
models_to_test = [
    {
        "id": "Qwen/Qwen3.5-9B",
        "gguf_repo": "lmstudio-community/Qwen3.5-9B-GGUF",
        "gguf_file": "Qwen3.5-9B-Q4_K_M.gguf",
    }
]

NUM_ITERATIONS_PER_TOPIC = 10
DELAY_BETWEEN_CALLS = 2.0
MAX_TOTAL_SECONDS = 30.0

benchmark_results = []


In [ ]:
try:
    competitions = client.competitions.list_all()
    print("\nFound topics to benchmark:")
    for c in competitions:
        print(f" - {c.name} (ID: {c.id})")
except Exception as e:
    print(f"Could not fetch competitions: {e}")
    competitions = []

for model_entry in models_to_test:
    model_id = model_entry['id']
    print(f"\n{'=' * 70}\nTesting model: {model_id}\n{'=' * 70}")
    try:
        print(f"Downloading {model_entry['gguf_file']}...")
        from huggingface_hub import hf_hub_download
        import os
        os.makedirs('/kaggle/working/gguf_models', exist_ok=True)
        gguf_path = hf_hub_download(repo_id=model_entry["gguf_repo"], filename=model_entry["gguf_file"], local_dir='/kaggle/working/gguf_models')
        from llama_cpp import Llama
        llm = Llama(model_path=str(gguf_path), n_gpu_layers=-1, n_ctx=4096, n_threads=4, verbose=False, seed=42)
    except Exception as e:
        print(f"Failed to load model {model_id}: {e}\nSkipping...\n")
        continue

    model_overall_correct = 0
    model_overall_wrong = 0
    model_overall_timeouts = 0
    model_overall_earnings = 0
    question_logs = []
    model_answer_times = []
    topic_stats = {}

    for comp in competitions:
        if comp.id != 3: continue
        print(f"\n  >> Starting Topic: {comp.name} (ID: {comp.id})")
        comp_correct = 0
        comp_wrong = 0
        comp_timeouts = 0
        comp_answer_times = []

        for i in range(NUM_ITERATIONS_PER_TOPIC):
            print(f"     Game {i + 1}/{NUM_ITERATIONS_PER_TOPIC}... ", end="")
            time.sleep(DELAY_BETWEEN_CALLS)
            try:
                game = client.game.start(competition_id=comp.id)
            except Exception as e:
                print(f'Start error: {e}, retrying...')
                time.sleep(3)
                continue

            while game.in_progress:
                question = game.current_question
                if not question: break
                question_received_at = time.perf_counter()

                decision = choose_math_answer_with_tools(question, llm)

                answer_id = decision["answer_id"]
                submitted_at = time.perf_counter()
                answer_time_seconds = submitted_at - question_received_at

                print(
                    f"answer={answer_id} "
                    f"source={decision.get('source')} "
                    f"tools={decision.get('tools_used', [])} "
                    f"conf={decision.get('confidence', 0.0):.2f} "
                    f"time={answer_time_seconds:.2f}s"
                )

                comp_answer_times.append(answer_time_seconds)
                model_answer_times.append(answer_time_seconds)
                time.sleep(DELAY_BETWEEN_CALLS)

                try:
                    result = game.answer(answer_id)
                except Exception as e:
                    print(f'Answer error: {e}, skipping question...')
                    time.sleep(3)
                    break
                options_dict = get_options_dict(question)
                question_logs.append({
                    "Topic": comp.name,
                    "Question": question.text,
                    "Options": json.dumps(options_dict),
                    "Generated Code": decision.get("generated_code", ""),
                    "Script Output": decision.get("script_output", ""),
                    "Raw Final LLM Output": decision.get("raw_output", ""),
                    "Our Answer ID": answer_id,
                    "Our Answer Text": options_dict.get(answer_id, ""),
                    "Correct": result.correct,
                    "Time Taken (s)": answer_time_seconds,
                    "API Raw Response": str(result)
                })
                if result.correct:
                    comp_correct += 1
                    if result.game_over: model_overall_earnings += result.earned_amount
                elif result.timed_out:
                    comp_timeouts += 1
                    model_overall_earnings += result.earned_amount
                else:
                    comp_wrong += 1
                    model_overall_earnings += result.earned_amount
            print("Done.")

        topic_stats[comp.name] = {
            "correct": comp_correct,
            "wrong": comp_wrong,
            "timeouts": comp_timeouts,
            "answer_times": comp_answer_times,
        }
        model_overall_correct += comp_correct
        model_overall_wrong += comp_wrong
        model_overall_timeouts += comp_timeouts

    overall_total = model_overall_correct + model_overall_wrong + model_overall_timeouts
    overall_acc = (model_overall_correct / overall_total * 100) if overall_total > 0 else 0

    flat_result = {
        "Model": model_id,
        "Overall Accuracy (%)": round(overall_acc, 2),
        "Total Earnings ($)": model_overall_earnings,
        "Avg Time / Question (s)": round(statistics.mean(model_answer_times), 3) if model_answer_times else None,
    }

    for t_name, t_stats in topic_stats.items():
        t_total = t_stats["correct"] + t_stats["wrong"] + t_stats["timeouts"]
        t_acc = (t_stats["correct"] / t_total * 100) if t_total > 0 else 0
        times = t_stats["answer_times"]
        flat_result[f"{t_name} Accuracy (%)"] = round(t_acc, 2)
        flat_result[f"{t_name} Avg Time / Question (s)"] = round(statistics.mean(times), 3) if times else None

    benchmark_results.append(flat_result)
    pd.DataFrame(question_logs).to_csv(f'/kaggle/working/question_logs_{model_id.replace("/", "_")}.csv', index=False)
    print(f"\nSaved detailed question logs for {model_id}!")
    del llm
    gc.collect()
    torch.cuda.empty_cache()

print("\nBenchmarking Complete!")


In [ ]:
import pandas as pd

df_results = pd.DataFrame(benchmark_results)
if not df_results.empty:
    display(df_results)
    output_csv = "/kaggle/working/benchmark_results.csv"
    df_results.to_csv(output_csv, index=False)
    print(f"\nSaved final results to: {output_csv}")

# Optional local evaluation on CSV (if present)
csv_path = "/kaggle/input/math_samples.csv"
if os.path.exists(csv_path):
    print(f"\nFound local math samples at {csv_path}, running batch evaluation...")
    df = pd.read_csv(csv_path, sep=None, engine='python')
    # This block can be expanded to run tools on the DF
    print("Batch evaluation logic goes here.")
else:
    print("\nNo local CSV samples found. Skipping batch evaluation.")


## 9.3 The "Thinking" Pipeline

During our experiments with the Maths section, we also explored a **"Thinking" Pipeline** designed to encourage explicit reasoning before code generation. In this setup, we modified the prompt to instruct the model to: *"First, think step-by-step about the mathematical logic required to solve this problem. Then, output your python script inside a ` ```python ` codeblock."*

**How it works:**
The orchestrator (`llm_generate_python`) prompts the model to generate a long Chain-of-Thought (CoT) reasoning block before it writes the executable Python script. The orchestrator then parses out the python codeblock, runs it securely in the background, and feeds the execution output into the final solver.

**Why it was abandoned:**
While theoretically powerful, this pipeline was highly problematic in practice, and we reverted to the direct non-thinking model for our core analysis. 

1. **Severe Timeout Issues (Primary Reason):** The PoliMillionaire game server enforces a strict 30-second timeout per question. Generating hundreds of "thinking" tokens before getting to the actual Python script almost always caused the pipeline to exceed this budget, resulting in constant timeouts. 
2. **Performance Degradation:** When the pipeline *did* manage to finish in time, the thinking model often performed worse than the direct non-thinking model. The extensive reasoning seemed to cause the model to lose track of the strict formatting constraints, leading to syntax errors or non-executable outputs. 

Given more time, we would have loved to explore the exact reasons behind this performance drop and potentially optimize the parsing logic or prompt structure. However, the frequent timeouts on our setup meant we had to skip straight to the non-thinking model for our main analysis.

Below is the implementation of the thinking orchestrator for reference.


In [ ]:
# --- PAL Pipeline Orchestrator ---
PYTHON_GENERATOR_SYSTEM_PROMPT = """You are an expert Python programmer solving math multiple-choice problems.
Write a short, efficient Python script using `math`, `statistics`, or `sympy` (imported as `sp`) to solve the problem. DO NOT output any conversational text or markdown before or after the code.
You MUST assign your final answer to a variable named 'ans' AND print(ans) at the end. Do not print intermediate steps.

CRITICAL RULES:
1. If using `sp.solve()`, it returns a list. Check for emptiness: `ans = sol[0] if sol else None`. Do not pass lists to `sp.Eq()`.
2. ABSTRACT ALGEBRA: For Modular Arithmetic or Polynomial Rings (e.g. Z_n), DO NOT use SymPy ring generators. Use basic Python integers and apply modulo `% n` to the final result.
3. INEQUALITIES: SymPy solve() fails on inequalities (e.g. |x| < 5). If asking for integers, write a basic Python `for` loop to check integers in a reasonable range.
4. STATISTICS & PROBABILITY: Prefer `statistics` module and `math.comb()`. For word problems: "A and B" = joint P(A ∩ B); "A given B" = conditional P(A|B); "no A" = 1 - P(A). Do NOT multiply probabilities unless independence is stated. Use P(A ∩ B) = P(A) - P(A ∩ ¬B) when applicable.
5. THEORETICAL QUESTIONS: If the problem requires NO calculations and is purely conceptual, STILL WRITE A PYTHON SCRIPT. Just use `print("your conceptual answer")`.
6. ZERO COMMENTS ALLOWED: Do NOT write ANY python comments (`#`) in your code. Writing comments causes a FATAL TIMEOUT. Output ONLY the raw code.
7. INFINITE SERIES: Prefer symbolic closed forms (e.g. sum x^k = 1/(1-x), sum kx^k = x/(1-x)^2) over numerical loops. Avoid while loops for infinite sums.

Example:
Problem: If 2x + 5 = 15, what is x?
Options:
0: 5
1: 10
2: 2

```python
import sympy as sp
x = sp.symbols('x')
eq = sp.Eq(2*x + 5, 15)
sol = sp.solve(eq, x)
ans = sol[0] if sol else None
print(ans)
```
"""

FAST_SOLVER_SYSTEM_PROMPT = """You are a fast, highly accurate math multiple-choice solver.
Use the output of the executed Python script to select the correct option.
WARNING: The script output might be an error or a partial step. Use your reasoning to select the final option. Do not show your work. Return ONLY JSON.
CRITICAL RULE: Never output the computed numeric value as the final answer ID! Always map the computed value to the matching option index. If the computed value is 2, and Option 1 is "2", you MUST return answer 1, not 2.
If no option exactly matches the script output, do not blindly guess the nearest option unless confidence is low.

Example JSON output:
{"answer": 2, "confidence": 0.95}
"""

def llm_generate_python(question, llm, rag_hint=""): 
    user_msg = f"Problem:\n{question.text}{rag_hint}\n\nFirst, think step-by-step about the mathematical logic required to solve this problem. Then, output your python script inside a ```python codeblock."
    prompt = f"<|im_start|>system\n{PYTHON_GENERATOR_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"
    print("  [LLM] Generating Python Code...")
    t_llm0 = time.perf_counter()
    try:
        outputs = llm.create_completion(prompt=prompt, max_tokens=800, temperature=0.0, stop=["<|im_end|>"])
    except Exception as e:
        print("Fallback LLM error:", e)
        return "", "Error"
    t_llm1 = time.perf_counter()
    print(f"  [LLM] Code Generation took {t_llm1 - t_llm0:.2f}s")
    text = outputs["choices"][0]["text"] or ""
    text = text.strip()
    
    if "SKIP" in text[:30]:
        return "SKIP", text
    
    m = re.search(r"```python\n(.*?)(?:```|$)", text, re.DOTALL)
    if m: return m.group(1).strip(), text
    
    m2 = re.search(r"```(.*?)```", text, re.DOTALL)
    if m2: return m2.group(1).strip(), text
    
    lines = text.split("\n")
    code_lines = []
    for line in lines:
        if line.strip().startswith(("import ", "from ", "def ", "ans =", "print(", "x =", "y =", "z =", "eq =", "sol =")):
            code_lines.append(line)
        elif code_lines and not line.strip().startswith(("The ", "Let ", "Here ", "Since ", "Therefore ")):
            code_lines.append(line)
            
    if code_lines:
        return "\n".join(code_lines).strip(), text
    return "", text
def build_pipeline_prompt(question, script_output, option_check, rag_hint=""):
    options = get_options_dict(question)
    prompt = f"Solve this math multiple-choice problem quickly and accurately.\n\nProblem:\n{question.text}{rag_hint}\n\nOptions:\n"
    for opt_id, opt_text in options.items(): prompt += f"{opt_id}: {opt_text}\n"
    
    if "Error:" in script_output or "No printed output" in script_output or script_output.strip() == "":
        pass # Hide syntax errors from Qwen to prevent confusion
    elif script_output == "SKIP":
        pass # It's a theoretical question, no tool output
    else:
        prompt += f"\nPython Script Execution Output:\n{script_output}\n"
        if option_check["confidence"] > 0:
            prompt += f"\nNote: The script output perfectly matches Option {option_check['answer_id']}. Verify if this is the final answer or just a partial step!\n"
    prompt += "\nReturn only JSON:\n{\"answer\": 0, \"confidence\": 0.0}\n"
    return prompt.strip()

def choose_math_answer_with_tools(question, llm):
    start = time.perf_counter()
    
    # --- RAG RETRIEVAL ---
    rag_context, matched_probs = retrieve_rag_context(question.text, top_k=3)
    rag_hint = f"\n\nHint: Here are some highly similar problems and their solutions:\n\n{rag_context}\n\nUse the mathematical logic from these examples to help solve the current problem." if rag_context else ""
    # ----------------------
    
    # 1. LLM Python Code Generation
    code_str, raw_code_text = llm_generate_python(question, llm, rag_hint=rag_hint)
    
    # 2. Execute Python Code
    elapsed_after_gen = time.perf_counter() - start
    
    if code_str == "SKIP":
        script_output = "SKIP"
    elif elapsed_after_gen > 24.0:
        script_output = "Timeout: Code generation took too long. Proceeding without tools."
    elif code_str:
        exec_timeout = max(0.5, min(6.0, 25.0 - elapsed_after_gen))
        print("  [EXEC] Running Python Script...")
        t_ex0 = time.perf_counter()
        script_output = execute_python_code(code_str, timeout=exec_timeout)
        t_ex1 = time.perf_counter()
        print(f"  [EXEC] Script execution took {t_ex1 - t_ex0:.2f}s")
    else:
        script_output = "No valid Python code generated."
        
    # 3. Option Checker
    if script_output in ["SKIP", "Timeout: Code generation took too long. Proceeding without tools.", "No valid Python code generated."]:
        option_check = {"answer_id": 0, "confidence": 0.0, "text": "", "reason": "No matches"}
    else:
        option_check = tool_option_checker(question, script_output)
        
    if option_check["confidence"] >= 0.9:
        return {
            "generated_code": code_str,
            "script_output": script_output,
            "raw_code_text": raw_code_text,
            "answer_id": option_check["answer_id"],
            "confidence": option_check["confidence"],
            "source": "pal_early_exit",
            "raw_output": "Early Exit matched Option.",
            "elapsed": time.perf_counter() - start,
            "tools_used": ["python_executor"],
            "rag_matches": " || ".join(matched_probs)
        }
    # 4. LLM Answer Generator
    elapsed = time.perf_counter() - start
    remaining = max(3.0, min(12.0, 27.0 - elapsed))
    max_toks = max(10, min(128, int(remaining * 30)))
    
    # We can also use create_completion here to force JSON output
    user_msg2 = build_pipeline_prompt(question, script_output, option_check)
    prompt2 = f"<|im_start|>system\n{FAST_SOLVER_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_msg2}<|im_end|>\n<|im_start|>assistant\n{{\n"
    print("  [LLM] Generating Answer...")
    t_llm0 = time.perf_counter()
    try:
        outputs = llm.create_completion(prompt=prompt2, max_tokens=max_toks, temperature=0.0, stop=["<|im_end|>"])
        text = "{" + (outputs["choices"][0]["text"] or "").strip()
    except Exception:
        text = "{}"
    t_llm1 = time.perf_counter()
    print(f"  [LLM] Answer Generation took {t_llm1 - t_llm0:.2f}s")
    
    ans_id = extract_answer(text)
    conf = extract_confidence(text)
    
    if conf < 0.1 and option_check["confidence"] > 0:
        ans_id = option_check["answer_id"]
        conf = 0.5
        text = "LLM failed format, fallback to option check. " + text
        
    return {
        "generated_code": code_str,
        "script_output": script_output,
        "raw_code_text": raw_code_text,
        "answer_id": ans_id,
        "confidence": conf,
        "source": "pal_with_qwen",
        "raw_output": text,
        "elapsed": time.perf_counter() - start,
        "tools_used": ["python_executor"] if code_str and code_str != "SKIP" else [],
        "rag_matches": " || ".join(matched_probs)
    }


# 10. Attempt to Fine-tune Qwen3-8B for Mathematical Reasoning: Approaches and Results 💡

## 10.1 Context

The goal was to obtain a specialized model for mathematical multiple-choice reasoning, deployable via `llama.cpp` for the **PoliMillionaire** benchmark (category "Maths"). The chosen base model is **Qwen3-8B**, fine-tuned with LoRA to reduce training costs and preserve the ability to merge the learned weights back into the base model before GGUF quantization.

## 10.2. Training: LoRA on AQUA-RAT

### 10.2.1 Dataset choice

[**AQUA-RAT**](https://huggingface.co/datasets/deepmind/aqua_rat) (Algebra Question Answering with Rationales, Ling et al. 2017, DeepMind) is a dataset of **97,467 mathematical multiple-choice examples**, each structured as:

```
question:  <natural-language word problem>
options:   ["A) ...", "B) ...", "C) ...", "D) ...", "E) ..."]
rationale: <step-by-step reasoning in natural English>
correct:   <letter>
```

The key feature of AQUA-RAT, compared to other mathematical datasets such as GSM8K or MATH, is that each example includes a **rationale in prose**: an articulated chain of reasoning in natural language, with intermediate calculations shown explicitly. The dataset is therefore designed to teach the model a specific **output distribution**: prose rationale → "Answer is X".

### 10.2.2 Training setup

- **Base model**: `Qwen/Qwen3-8B` (non-instruct)
- **Dataset**: AQUA-RAT full train split, 97,467 examples
- **Method**: LoRA adapter
- **Wall-clock time**: 9 hours on a single GPU
- **Output**: LoRA adapter published as [`elmat234/qwen3-8b-aqua-lora`](https://huggingface.co/elmat234/qwen3-8b-aqua-lora), then merged into the base model and quantized to Q4_K_M for `llama.cpp` ([`elmat234/qwen3-8b-aqua-Q4_K_M-gguf`](https://huggingface.co/elmat234/qwen3-8b-aqua-Q4_K_M-gguf), ~5 GB single file).

### 10.2.3 What the model actually learned

The fine-tuning aligned the model to produce outputs in the exact style of the training set. Concretely:

1. It receives a question + options labelled A–E
2. It emits a step-by-step rationale in prose
3. It ends with a string in the form `Answer is X`

This matters because *it determines which kind of inference pipeline can extract value from the model*. The AQUA LoRA did **not** learn to write Python code, did **not** learn to solve problems without seeing the options first, did **not** learn to invoke external tools. 

## 10.3. Inference pipelines attempted

Pipelines were tested on the **Maths** category of the PoliMillionaire benchmark (`competition_id=3`), with 10 games per pipeline.

### 10.3.1 Iteration 1: PAL (Program-Aided Language Model)

**Strategy.** PAL: the model generates Python code that is executed deterministically, and the returned value is matched against the numerical options. A two-phase pipeline:

1. Phase 1: code generation + execution in a CPU sandbox with `math/statistics/sympy`
2. Phase 2: if the output does not exactly match any option, a second LLM pass selects the option given the four alternatives and the script output

This pipeline was inherited from the benchmark of previous (non-LoRA) models, tuned for vanilla Qwen3.

**Run 1 (initial version, baseline)**

| Metric | Value |
|---|---|
| Maths Accuracy | **47.37%** |
| Avg time | 11.85s/q |
| Code execution errors | 16/19 (84%) |

**Diagnosis.** In Qwen3's native "thinking" mode, the model emitted `<think>...</think>` blocks before responding. The code extractor was taking the entire output and passing it to `exec()` the 1500+ characters of prose inside the `<think>` block caused `SyntaxError: unterminated string literal`. Furthermore, even when code came out clean, it contained Unicode math characters (`²`, `π`, `√`) which Python rejects.

**Fixes applied.** Three layers of defense:
1. `/no_think` soft switch in the user message (the only available approach from `llama-cpp-python`, which does not expose `chat_template_kwargs`)
2. `_strip_thinking()` to remove any residual `<think>` blocks
3. Robust code extractor: first looks for the ```` ```python ```` fence, then falls back to identifying the first code-like line and everything after
4. Anti-prose filter: discards outputs with too many English words relative to Python operators

**Run 2 (PAL fixed)**

| Metric | Value |
|---|---|
| Maths Accuracy | **75.61%** |
| Avg time | 3.21s/q |
| Code execution errors | 2/41 (5%) |
| Total earnings | $258,000 |

A net improvement, but **a result almost equivalent to what was already achievable with vanilla Qwen3.5-9B** on the same pipeline. The AQUA LoRA was not contributing positively: the pipeline was forcing it to write code, a modality it had never been trained on.

## 10.4. Abandoned attempt: fine-tuning on NuminaMath-CoT

### 10.4.1 Motivation

After verifying that AQUA-RAT is too narrow (predominantly algebra word problems) to cover the variety of domains in the PoliMillionaire benchmark (which includes geometry, probability, abstract algebra, group theory, basic analysis), we considered a second training run on a broader and more modern dataset.

[**NuminaMath-CoT**](https://huggingface.co/datasets/AI-MO/NuminaMath-CoT) is a dataset of approximately **860k mathematical problems** with Chain-of-Thought solutions, covering:

- Math competitions (AMC, AIME, olympiads)
- Standard university-level exercises
- Word problems
- Abstract algebra, geometry, number theory

A fine-tune on NuminaMath would plausibly have generalized better to the non-algebraic PoliMillionaire topics, where AQUA-RAT is effectively out of distribution.

### 10.4.2 Why it was not done

**GPU constraint.** A LoRA fine-tune of Qwen3-8B on a dataset of 860k examples requires an order-of-magnitude increase over the 97k of AQUA-RAT (~9× more data). At equal setup, the training run would have stretched from ~9 hours to well above 81 hours on a single consumer GPU, without accounting for the typically longer context length of NuminaMath problems, which further degrades throughput.

Without access to multi-GPU or enterprise-grade infrastructure (H100/A100) within the project's timeline, this option was shelved.





---
# 11. General Topics (Entertainment, History, Science, Philosophy) 📚

For all other categories except Maths and News, we initially relied on **Wikipedia** as our primary knowledge source. 

Our approach was to use the LLM to generate targeted search keywords based on the question. These keywords were then fed into a Wikipedia search to retrieve the most relevant pages. Finally, we used a **BM25 algorithm** to chunk and extract the precise context needed from those Wikipedia pages to help the model answer the question correctly.

*(Note: We also tried using this Wikipedia BM25 approach for the **News** section, but it did not perform well because News questions often cite highly specific, recent documents that aren't well-captured by Wikipedia. However, for these general knowledge topics, the BM25 Wikipedia approach was consistently the better version, so we stuck with it here.)*



## 📚 Wikipedia BM25 Tool

In [ ]:
import requests
import urllib.parse
from rank_bm25 import BM25Okapi
import string
import time

def wikipedia_search_bm25(query, options_text, llm_query, top_k_chunks=10):
    try:
        t0 = time.perf_counter()
        encoded_query = urllib.parse.quote(llm_query)
        search_url = f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={encoded_query}&utf8=&format=json&srlimit=3"
        headers = {'User-Agent': 'MillionaireNewsBench_Kaggle/1.0 (research@example.com)'}
        
        print("  [wiki_bm25] --- API CALL DETAILS ---")
        print("  [wiki_bm25] Request Type: HTTP GET (Raw requests library)")
        print(f"  [wiki_bm25] Raw Search String: {llm_query}")
        print(f"  [wiki_bm25] Exact URL Called: {search_url}")
        print("  [wiki_bm25] ------------------------")
        
        res = requests.get(search_url, headers=headers, timeout=10)
        t1 = time.perf_counter()
        print(f"  [wiki_bm25] API search took {t1 - t0:.2f}s")
        
        if res.status_code != 200:
            return f"Wikipedia HTTP Error: {res.status_code} {res.text[:100]}"
            
        data = res.json()
        search_results = data.get("query", {}).get("search", [])
        if not search_results:
            return "No Wikipedia articles found."
            
        paragraphs = []
        for item in search_results:
            title = item["title"]
            page_url = f"https://en.wikipedia.org/w/api.php?action=query&prop=extracts&explaintext=1&titles={urllib.parse.quote(title)}&format=json"
            page_res = requests.get(page_url, headers=headers, timeout=10)
            if page_res.status_code == 200:
                page_data = page_res.json()
                pages = page_data.get("query", {}).get("pages", {})
                for page_id, page_info in pages.items():
                    extract = page_info.get("extract", "")
                    page_paragraphs = [p.strip() for p in extract.split('\n\n') if len(p.strip()) > 50]
                    for p in page_paragraphs:
                        paragraphs.append(f"[{title}] {p}")
                        
        t2 = time.perf_counter()
        print(f"  [wiki_bm25] Page fetch/split took {t2 - t1:.2f}s")
                    
        if not paragraphs:
            return "Could not extract content from articles."
            
        def tokenize(text):
            text = text.lower()
            text = text.translate(str.maketrans('', '', string.punctuation))
            return text.split()
            
        tokenized_corpus = [tokenize(p) for p in paragraphs]
        bm25 = BM25Okapi(tokenized_corpus)
        
        search_query_full = f"{query} {options_text}"
        tokenized_query = tokenize(search_query_full)
        top_chunks = bm25.get_top_n(tokenized_query, paragraphs, n=top_k_chunks)
        
        t3 = time.perf_counter()
        print(f"  [wiki_bm25] BM25 ranking took {t3 - t2:.2f}s")
        
        compact_text = ""
        for i, chunk in enumerate(top_chunks):
            compact_text += f"• {chunk}\n\n"
            if len(compact_text) > 4500:  # Prevent massive context bloat
                break
            
        return compact_text.strip()[:5000]
    except Exception as e:
        return f"Wikipedia Error: {e}"

WIKI_QUERY_SYSTEM_PROMPT = """You are an expert Wikipedia search query generator. 
Extract EXACTLY 2 to 4 keywords from the question.
Example: "When did Michael O'Leary talk about jet fuel?" -> "Michael O'Leary jet fuel"
CRITICAL: Output ONLY the 2-4 keywords and absolutely nothing else. No punctuation, no conversational text."""

def llm_generate_wiki_query(question, llm):
    t0 = time.perf_counter()
    options = get_options_dict(question)
    options_text = " OR ".join([str(v) for v in options.values()])
    prompt_user = f"Problem:\n{question.text}\nOptions:\n{options_text}\n\nGenerate search keywords:"
    prompt = f"<|im_start|>system\n{WIKI_QUERY_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{prompt_user}<|im_end|>\n<|im_start|>assistant\n"
    
    try:
        outputs = llm.create_completion(prompt=prompt, max_tokens=20, temperature=0.0, stop=["<|im_end|>"])
        text = outputs["choices"][0]["text"].strip()
    except Exception as e:
        print("LLM Error:", e)
        text = "Error"
        
    # Safety fallback if LLM generates a full sentence
    if len(text.split()) > 6:
        text = " ".join(text.split()[:5])
    t1 = time.perf_counter()
    print(f"  [wiki_llm] LLM Query Gen took {t1 - t0:.2f}s")
    return text

---
# 12. News (Live Search Integration) 📰

For the **News** category, static databases are insufficient. We needed live access to the web, so we integrated two powerful search APIs: **Exa** and **Firecrawl**.

1. **Exa**: We chose Exa because it offers a highly optimized search service designed specifically for LLMs. While the paid version can return vectors directly, the free version is still incredibly powerful, featuring its own built-in `highlights` extraction that functions similarly to a BM25 chunk extractor to pull the most relevant snippets.
2. **Firecrawl**: We also experimented with Firecrawl. Although we used their hosted API for simplicity, Firecrawl is an open-source service that we could easily run locally via a Docker image, giving us access to all premium features for free. One of its standout features is the ability to extract dates directly from the document metadata, which is crucial for News questions where the temporal context is everything. 

Both services performed exceptionally well, achieving **often 15/15 accuracy** in the News category. Below is the implementation for our live News solver.


In [ ]:
from kaggle_secrets import UserSecretsClient
from exa_py import Exa
import time
import re
import torch

try:
    user_secrets = UserSecretsClient()
    exa_api_key = user_secrets.get_secret("EXA_KEY")
    exa = Exa(exa_api_key)
    print("Exa client initialized via Kaggle Secrets!")
except Exception as e:
    print("Could not initialize Exa via Kaggle Secrets. Using provided fallback.")
    exa = Exa("********")

def format_exa_results(exa_result):
    if not getattr(exa_result, 'results', None):
        return "No results."
    compact_text = ""
    for i, res in enumerate(exa_result.results): 
        date = res.published_date[:10] if res.published_date else "Unknown Date"
        author = res.author if getattr(res, 'author', None) else "Unknown Author"
        compact_text += f"• {res.title} ({date}) - {author}\n"
        if res.highlights:
            for hl in res.highlights:
                hl_clean = re.sub(r'\s+', ' ', hl).strip()
                compact_text += f" - {hl_clean}\n"
    return compact_text.strip()

In [ ]:
import re
import os
import requests
from kaggle_secrets import UserSecretsClient

def firecrawl_search(query):
    url = "https://api.firecrawl.dev/v2/search"
    
    payload = {
      "query": query,
      "sources": [
        "web",
        "news"
      ],
      "categories": [],
      "limit": 10,
      "scrapeOptions": {
        "onlyMainContent": False,
        "maxAge": 172800000,
        "timeout": 20000,
        "parsers": [
          "pdf"
        ],
        "formats": []
      }
    }
    
    date_match = re.search(r'(\d{4})-(\d{2})-(\d{2})', query)
    if date_match:
        year, month, day = date_match.groups()
        date_str = f"{int(month)}/{int(day)}/{year}"
        payload["tbs"] = f"cdr:1,cd_min:{date_str},cd_max:{date_str}"
    
    try:
        user_secrets = UserSecretsClient()
        fc_key = user_secrets.get_secret("FC_KEY")
    except:
        fc_key = os.environ.get("FC_KEY", "")
        
    headers = {
        "Authorization": f"Bearer {fc_key}",
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(url, json=payload, headers=headers)
        if response.status_code == 200:
            data = response.json()
            
            compact_text = ""
            
            response_data = data.get("data", data)
            
            # Extract News
            for item in response_data.get("news", []):
                if not isinstance(item, dict): continue
                title = item.get("title", "")
                snippet = item.get("snippet", "")
                
                compact_text += f"• {title}\n"
                if snippet:
                    compact_text += f" - {snippet}\n"
                    
            # Extract Web
            for item in response_data.get("web", []):
                if not isinstance(item, dict): continue
                title = item.get("title", "")
                desc = item.get("description", "")
                markdown = item.get("markdown", "")
                
                compact_text += f"• {title}\n"
                
                text_content = ""
                if desc: text_content = desc
                elif markdown and isinstance(markdown, str): text_content = markdown[:300] + "..."
                
                if text_content:
                    compact_text += f" - {text_content}\n"
            
            return compact_text.strip() if compact_text else "No results."
        else:
            return f"Firecrawl Error: {response.status_code} {response.text}"
    except Exception as e:
        return f"Firecrawl Request failed: {e}"


In [ ]:
NEWS_SOLVER_SYSTEM_PROMPT = """You are an expert news researcher. You are given a multiple-choice question and some search results containing recent news.
Use the search results to find the correct answer to the question.
CRITICAL RULE: Return ONLY JSON in the following format:
{"answer": 0, "confidence": 0.95}
Do not return any other text. Map your answer to the correct option index."""

def choose_news_answer(question, tokenizer, model, exa_client, search_source="exa"):
    start = time.perf_counter()
    
    options = get_options_dict(question)
    options_text = " OR ".join([str(v) for v in options.values()])
    query = f"{question.text} {options_text}"
    
    search_context = ""
    tools_used = []
    
    if search_source == "exa":
        try:
            res = exa_client.search(
                query,
                num_results=10,
                type="fast",
                contents={"highlights": {"max_characters": 500}}
            )
            search_context += "--- EXA SEARCH RESULTS ---\n" + format_exa_results(res) + "\n\n"
            tools_used.append("exa_search")
        except Exception as e:
            search_context += f"--- EXA SEARCH RESULTS ---\nSearch failed: {e}\n\n"
            
    elif search_source == "firecrawl":
        try:
            fc_results = firecrawl_search(query)
            search_context += "--- FIRECRAWL SEARCH RESULTS ---\n" + fc_results + "\n\n"
            tools_used.append("firecrawl_search")
        except Exception as e:
            search_context += f"--- FIRECRAWL SEARCH RESULTS ---\nSearch failed: {e}\n\n"
            
    prompt = f"Problem:\n{question.text}\n\nOptions:\n"
    for opt_id, opt_text in options.items(): prompt += f"{opt_id}: {opt_text}\n"
    
    prompt += f"\nSearch Results:\n{search_context}\n\nReturn only JSON:\n{{\"answer\": 0, \"confidence\": 0.0}}\n"
    
    messages = [
        {"role": "system", "content": NEWS_SOLVER_SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    
    try: inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt", enable_thinking=False).to(model.device)
    except: inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
        
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        
    text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    ans_id = extract_answer(text)
    conf = extract_confidence(text)
    
    torch.cuda.empty_cache()
    
    return {
        "generated_code": "",
        "script_output": search_context.strip(), 
        "raw_code_text": "",
        "answer_id": ans_id,
        "confidence": conf,
        "source": search_source,
        "raw_output": text,
        "elapsed": time.perf_counter() - start,
        "tools_used": tools_used
    }

## News Pipeline Configuration & Model Reloading
*Note: At the end of the Maths section, we completely cleared the VRAM to ensure we had a clean slate. Because we are now moving to the News pipeline which uses the HuggingFace model directly, we must re-initialize the model and tokenizer here. This is a common pattern when working with large models on a single GPU!*

In [ ]:
models_to_test = ["Qwen/Qwen3.5-9B"]
NUM_ITERATIONS_PER_TOPIC = 5
DELAY_BETWEEN_CALLS = 2.0
MAX_TOTAL_SECONDS = 30.0

# Global list to hold logs from both runs
all_question_logs = []

model_id = models_to_test[0]
print(f"\nLoading model: {model_id}")
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    dtype=torch.float16, 
    device_map="auto", 
    quantization_config=quantization_config, 
    trust_remote_code=True
)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
print("Model reloaded successfully!")

In [ ]:
def run_benchmark(search_source):
    print(f"\n{'=' * 70}\nStarting Benchmark with {search_source.upper()}\n{'=' * 70}")
    
    try:
        competitions = client.competitions.list_all()
    except Exception as e:
        print(f"Could not fetch competitions: {e}")
        return

    comp = next((c for c in competitions if c.id == 5), None)
    if not comp:
        print("News competition (ID 5) not found!")
        return

    print(f"\n  >> Starting Topic: {comp.name} (ID: {comp.id})")

    for i in range(NUM_ITERATIONS_PER_TOPIC):
        print(f"     Game {i + 1}/{NUM_ITERATIONS_PER_TOPIC}... ", end="")
        time.sleep(DELAY_BETWEEN_CALLS)
        try:
            game = client.game.start(competition_id=comp.id)
        except Exception as e:
            print(f'Start error: {e}, retrying...')
            time.sleep(3)
            continue

        while game.in_progress:
            question = game.current_question
            if not question: break
            question_received_at = time.perf_counter()

            decision = choose_news_answer(question, tokenizer, model, exa, search_source=search_source)

            answer_id = decision["answer_id"]
            submitted_at = time.perf_counter()
            answer_time_seconds = submitted_at - question_received_at

            print(
                f"answer={answer_id} "
                f"source={decision.get('source')} "
                f"tools={decision.get('tools_used', [])} "
                f"conf={decision.get('confidence', 0.0):.2f} "
                f"time={answer_time_seconds:.2f}s"
            )

            time.sleep(DELAY_BETWEEN_CALLS)

            try:
                result = game.answer(answer_id)
            except Exception as e:
                print(f'Answer error: {e}, skipping question...')
                time.sleep(3)
                break
            
            options_dict = get_options_dict(question)
            all_question_logs.append({
                "Topic": comp.name,
                "Question": question.text,
                "Options": json.dumps(options_dict),
                "Script Output": decision.get("script_output", ""),
                "Raw Final LLM Output": decision.get("raw_output", ""),
                "Our Answer ID": answer_id,
                "Our Answer Text": options_dict.get(answer_id, ""),
                "Correct": result.correct,
                "Time Taken (s)": answer_time_seconds,
                "Search Source": search_source
            })
        print("Done.")

In [ ]:
run_benchmark(search_source="exa")


In [ ]:
run_benchmark(search_source="firecrawl")


In [ ]:
import pandas as pd

df_results = pd.DataFrame(all_question_logs)
if not df_results.empty:
    display(df_results)
    output_csv = "/kaggle/working/question_logs_all.csv"
    df_results.to_csv(output_csv, index=False)
    print(f"\nSaved final results with search sources to: {output_csv}")
else:
    print("No logs collected.")


---
# 13. Live Search on General Topics 🌍

While the BM25 Wikipedia approach was our primary method for the general topics (Entertainment, Ancient History, Science & Nature, Philosophy & Psychology), we also tested our **Live Search Pipeline (Exa / Firecrawl)** on these subjects.

Because live search engines index massive amounts of data beyond just Wikipedia, the pipeline performs exceptionally well here too, frequently achieving a perfect **15/15** score across these categories!

Below is the boilerplate to run the live search pipeline against all competitions except Maths (3) and News (5).


In [ ]:
all_general_logs = []

def run_benchmark_general(search_source):
    print(f"\n{'=' * 70}\nStarting General Benchmark with {search_source.upper()}\n{'=' * 70}")
    
    try:
        competitions = client.competitions.list_all()
    except Exception as e:
        print(f"Could not fetch competitions: {e}")
        return

    # Filter out Maths (3) and News (5)
    comps_to_run = [c for c in competitions if c.id not in [3, 5]]
    
    if not comps_to_run:
        print("No general competitions found!")
        return

    for comp in comps_to_run:
        print(f"\n  >> Starting Topic: {comp.name} (ID: {comp.id})")

        for i in range(NUM_ITERATIONS_PER_TOPIC):
            print(f"     Game {i + 1}/{NUM_ITERATIONS_PER_TOPIC}... ", end="")
            import time
            time.sleep(DELAY_BETWEEN_CALLS)
            try:
                game = client.game.start(competition_id=comp.id)
            except Exception as e:
                print(f'Start error: {e}, retrying...')
                time.sleep(3)
                continue

            while game.in_progress:
                question = game.current_question
                if not question: break
                import time
                question_received_at = time.perf_counter()

                # We reuse choose_news_answer as the general live search solver
                decision = choose_news_answer(question, tokenizer, model, exa, search_source=search_source)

                answer_id = decision["answer_id"]
                submitted_at = time.perf_counter()
                answer_time_seconds = submitted_at - question_received_at

                print(
                    f"answer={answer_id} "
                    f"source={decision.get('source')} "
                    f"tools={decision.get('tools_used', [])} "
                    f"conf={decision.get('confidence', 0.0):.2f} "
                    f"time={answer_time_seconds:.2f}s"
                )

                time.sleep(DELAY_BETWEEN_CALLS)

                try:
                    result = game.answer(answer_id)
                except Exception as e:
                    print(f'Answer error: {e}, skipping question...')
                    time.sleep(3)
                    break
                
                options_dict = get_options_dict(question)
                import json
                all_general_logs.append({
                    "Topic": comp.name,
                    "Question": question.text,
                    "Options": json.dumps(options_dict),
                    "Script Output": decision.get("script_output", ""),
                    "Raw Final LLM Output": decision.get("raw_output", ""),
                    "Our Answer ID": answer_id,
                    "Our Answer Text": options_dict.get(answer_id, ""),
                    "Correct": result.correct,
                    "Time Taken (s)": answer_time_seconds,
                    "Search Source": search_source
                })
            print("Done.")


In [ ]:
# Run the general benchmark with Exa
run_benchmark_general(search_source="exa")

In [ ]:
# Run the general benchmark with Firecrawl
run_benchmark_general(search_source="firecrawl")

In [ ]:
# Run the general benchmark with Wikipedia BM25
run_benchmark_general(search_source="wikipedia_bm25")

In [ ]:
import pandas as pd

df_general_results = pd.DataFrame(all_general_logs)
if not df_general_results.empty:
    display(df_general_results)
    output_csv = "/kaggle/working/general_live_search_logs.csv"
    df_general_results.to_csv(output_csv, index=False)
    print(f"\nSaved general live search results to: {output_csv}")
else:
    print("No logs collected.")

# 14. Automatic Speech Recognition (ASR)

In this section, we present the implementation of the ASR module. For the speech recognition task, we implemented the **`openai/whisper-small`** model using the Hugging Face `transformers` `pipeline`. This specific model size was selected because it offers an optimal balance between reliable transcription accuracy and a lightweight hardware footprint. In this way, the time constraints still allowed us to respond in a fast way, leveraging the LLM in `.gguf` format, loaded using `llama.cpp`.

* **In-Memory Audio Processing:** The game's API returns audio as raw binary streams (`bytes`). Rather than introducing latency by writing these to the disk, we utilized `io.BytesIO` combined with `soundfile` (`sf.read`). This allows us to decode the binary data directly into NumPy arrays in RAM and feed the raw waveforms and sampling rates straight into the Whisper pipeline for near-instant transcription.
* **Strategic Timer Bypass:** The PoliMillionaire server enforces a strict 30-second timeout. To prevent the physiological computing time of the ASR model from eating into the LLM's reasoning budget, we decoupled the fetching process. The audio for the main question and the first three options (A, B, C) are requested and transcribed *while the server timer is still frozen*. The 30-second countdown only triggers when the final option (D) is fetched. By doing this, 90% of the transcription workload is completed off-the-clock.
* **Aggressive VRAM Management:** To ensure system stability over hundreds of benchmark iterations, we implemented explicit garbage collection (`gc.collect()`) and GPU cache clearing (`torch.cuda.empty_cache()`) at the end of every single question loop. This reliably prevents CUDA Out Of Memory (OOM) crashes.

The transcription capabilities proved to be highly positive and reliable. We did observe a minor anomaly where background noise or altered tones were occasionally transcribed as vocalizations or stutters (e.g., "ha ha ha ha ha"). However, thanks to the LLM's semantic resilience, this artifact did not negatively impact or impair the agent's ability to accurately read, parse, and answer the underlying questions.

In [ ]:
!pip install soundfile librosa

In [ ]:

import io, gc, time
import torch
import soundfile as sf
from transformers import pipeline as hf_pipeline

print("Loading Whisper-small ASR pipeline ...")
_asr_device = 0 if torch.cuda.is_available() else -1
stt_pipe = hf_pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=_asr_device,
)
print(f"  Whisper-small loaded on {'GPU' if _asr_device == 0 else 'CPU'}.")


# ---- 12.2  transcribe_audio() helper ----
def transcribe_audio(audio_bytes: bytes) -> str:
    """Decode raw audio bytes and return the Whisper transcript.

    Uses soundfile + io.BytesIO for in-memory decoding (no disk I/O).
    Returns an empty string on empty / unreadable input.
    """
    if not audio_bytes:
        return ""
    try:
        data, samplerate = sf.read(io.BytesIO(audio_bytes))
        result = stt_pipe({"raw": data, "sampling_rate": samplerate})
        return result["text"].strip()
    except Exception as exc:
        print(f"  [ASR:WARN] transcribe_audio failed: {exc}")
        return ""


# ---- 12.3  Speech-mode benchmark (mirrors §5 llama.cpp accuracy run) ----
import pandas as pd

# Reuse the same iteration count and delay defined in §4.
# `llm` must already be loaded (run §5 download + load_gguf cells first).

per_q_log_speech = []   # per-question telemetry, same schema as per_q_log_cpp
competitions = client.competitions.list_all()

for comp in competitions:
    print(f"\n{'='*60}\nTopic: {comp.name}\n{'='*60}")

    for i in range(NUM_ITERATIONS_PER_TOPIC):
        print(f"  Game {i + 1}/{NUM_ITERATIONS_PER_TOPIC} ... ", end="")
        time.sleep(DELAY_BETWEEN_CALLS)

        # --- start game in speech mode ----
        try:
            game = client.game.start(competition_id=comp.id, mode="speech")
        except TypeError:
            # Server does not accept `mode` as a keyword arg -> positional fallback
            print("[start-fallback] ", end="")
            game = client.game.start(comp.id)
            game.mode = "speech"
            try:
                if hasattr(game, "refresh_state"):
                    game.refresh_state()
            except Exception:
                pass
        except Exception as exc:
            print(f"Start error: {exc}, skipping game.")
            time.sleep(3)
            continue

        # --- question loop ----
        while game.in_progress:
            q = game.current_question
            if not q:
                break

            t_stt_start = time.perf_counter()

            # Phase A: transcribe question + options A/B/C (timer still frozen)
            try:
                q.text = transcribe_audio(game.fetch_audio_question())
                print(f"\n  [STT] Q: {q.text}")

                opt_audios = [game.fetch_audio_option_next() for _ in range(3)]
                for idx, audio_b in enumerate(opt_audios):
                    q.options[idx].text = transcribe_audio(audio_b)
                    print(f"  [STT] Option [{idx}]: {q.options[idx].text}")

                # Phase B: fetch option D — THIS starts the 30-second server timer
                print("  [TIMER] Fetching option D -> 30-second countdown starts now.")
                q.options[3].text = transcribe_audio(game.fetch_audio_option_next())
                print(f"  [STT] Option [3]: {q.options[3].text}")

                # Sync local game state now that the timer is running
                if hasattr(game, "refresh_state"):
                    game.refresh_state()

            except Exception as exc:
                print(f"\n  [ASR:ERROR] Audio fetch/transcription failed: {exc}")
                # Fall back to whatever text the question object already has

            t_stt = time.perf_counter() - t_stt_start
            print(f"  [STT] Transcription completed in {t_stt:.2f}s")

            # Phase C: answer with the GGUF pipeline (same as §5)
            try:
                ans_id, tel = gguf_answer(q, q.options, llm)
            except Exception as exc:
                print(f"  [AGENT:ERROR] gguf_answer failed: {exc}")
                ans_id, tel = 0, {
                    "t_logits": 0.0, "t_cot": 0.0,
                    "t_total": 0.0, "n_new": 0,
                    "tok_per_s_cot": 0.0,
                    "logits_gap": -1.0, "logits_found": 0,
                    "early_exit": False,
                }

            print(
                f"  [AGENT] answer={ans_id} "
                f"gap={tel.get('logits_gap', 0.0):.2f} "
                f"early_exit={tel.get('early_exit', False)} "
                f"t_agent={tel.get('t_total', 0.0):.2f}s"
            )

            time.sleep(DELAY_BETWEEN_CALLS)

            # Phase D: submit answer
            try:
                result = game.answer(ans_id)
            except Exception as exc:
                print(f"  [SERVER:ERROR] game.answer failed: {exc}")
                time.sleep(3)
                break

            verdict = (
                "✅ CORRECT" if result.correct
                else "⏰ TIMEOUT" if getattr(result, "timed_out", False)
                else "❌ WRONG"
            )
            print(f"  [RESULT] {verdict} | earned: ${result.earned_amount}")

            # Log entry — same fields as per_q_log_cpp for easy DataFrame concat
            per_q_log_speech.append({
                "model":        "whisper-small+gguf",
                "topic":        comp.name,
                "iteration":    i,
                "level":        game.current_level,
                "question":     q.text,
                "chosen":       ans_id,
                "correct":      bool(result.correct),
                "timed_out":    bool(getattr(result, "timed_out", False)),
                "t_stt":        round(t_stt, 3),
                "t_logits":     round(tel.get("t_logits", 0.0), 3),
                "t_cot":        round(tel.get("t_cot", 0.0), 3),
                "t_total":      round(tel.get("t_total", 0.0), 3),
                "logits_gap":   round(tel.get("logits_gap", -1.0), 4),
                "logits_found": tel.get("logits_found", 0),
                "early_exit":   tel.get("early_exit", False),
                "n_new":        tel.get("n_new", 0),
                "tok_per_s":    round(tel.get("tok_per_s_cot", 0.0), 2),
                "earned":       result.earned_amount,
            })

            # Aggressive VRAM cleanup after each question to prevent CUDA OOM
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print("done.")

# ---- 12.4  Summarise results ----
df_speech = pd.DataFrame(per_q_log_speech)

if not df_speech.empty:
    total   = len(df_speech)
    correct = df_speech["correct"].sum()
    acc     = correct / total * 100

    print(f"\n=== ASR Speech-Mode Results ===")
    print(f"Overall accuracy : {acc:.2f}% ({correct}/{total})")
    print(f"Mean t_stt       : {df_speech['t_stt'].mean():.2f}s")
    print(f"Mean t_agent     : {df_speech['t_total'].mean():.2f}s")
    print(f"\nPer-topic accuracy:")
    print(
        df_speech.groupby("topic")["correct"]
        .agg(lambda s: f"{s.mean()*100:.1f}% ({s.sum()}/{len(s)})")
        .to_string()
    )

    out_csv = "/kaggle/working/asr_speech_results.csv"
    df_speech.to_csv(out_csv, index=False)
    print(f"\nSaved to: {out_csv}")
    display(df_speech)
else:
    print("No speech-mode logs collected.")


---
# 15. Confidence Estimation

During the optimization phase of our pipeline, we explored advanced techniques for **Confidence Estimation** to evaluate how reliably our model selects a specific answer. Specifically, we investigated the **Repeat Sampling** method. 

According to the literature, this technique involves prompting the LLM multiple times with the same query (or with minor semantic variations) to calculate the empirical frequency of the majority answer:

$$P(\text{answer} \mid \text{query}) \approx \frac{\text{Count}(\text{Answer } X)}{\text{Total Samples}}$$

A high concentration of votes on a single option signals elevated model confidence, whereas scattered outputs indicate semantic ambiguity or factual contention within the model's training data.

### Why Repeat Sampling Was Not Implemented

Despite its theoretical benefits, we ultimately decided not to develop the estimate block in our system due to the determinism of low temperature ($\text{Temperature} = 0.0$). Our current system relies on a quantized version of `Qwen-3.5-9B` running with greedy decoding (`temperature=0.0`) to maximize the stability and syntax accuracy of the generated PAL scripts. At zero temperature, the token selection is entirely deterministic, meaning the model will always choose the token with the highest log-probability at each step. Running multiple samples would inevitably yield the exact same code output $N$ times, creating a false illusion of $100\%$ confidence while wasting valuable computational resources.

For these reasons, since we could not loosen the deterministic constraints of our core math solver and other confidence estimates like Token-based Probability or Verbalized Confidence were either too difficult to implement or too stochastic, the confidence estimate of our model remained a theoretical evaluation and no code was developed for it in our final deployment.


# 16. Literature Review & Next Steps

## Literature Review

During the development of this research, we consulted the paper [*Beyond Semantic Similarity: Rethinking Retrieval for Agentic Search via Direct Corpus Interaction*](https://arxiv.org/pdf/2605.05242) to understand how we could bring our models one step further. The paper introduces **Direct Corpus Interaction (DCI)**, a retrieval paradigm arguing that agentic search systems should not rely only on a fixed top-*k* retrieval interface. Li et al. (2026) show that conventional sparse or dense retrievers can become a bottleneck when an agent needs to combine weak clues, enforce exact lexical constraints, inspect local context, or revise its search strategy after partial evidence. Instead, DCI gives the agent direct access to the raw corpus through simple tools such as `grep`, file reads, shell commands, and lightweight scripts. This creates a higher-resolution retrieval interface where the model can search, verify, and refine evidence step by step.

Our project was inspired by this idea in three main ways:

- **Tool-based retrieval over fixed retrieval alone.** In our live-search experiments, we compared external search sources such as Exa, Firecrawl, and Wikipedia/BM25 rather than relying only on the base model’s internal knowledge. This follows the DCI motivation that agents perform better when they can actively access external evidence.

- **Exact and local evidence checking.** The paper emphasizes that many agentic tasks require precise lexical matching and local context inspection. This connects to our benchmark design, where questions often depend on specific factual cues, topic knowledge, or answer-option distinctions that are not always captured by semantic similarity alone.

- **Practical agent constraints.** DCI also highlights the importance of runtime tradeoffs such as cost, latency, context size, and tool-call efficiency. These were central to our work as well, especially in our comparison of HuggingFace vs. `llama.cpp`, quantized GGUF models, throughput, VRAM usage, and fallback reasoning strategies.

## What We Would Have Liked to Implement

Given more time, we would have liked to extend our pipeline with a more DCI-style search and verification layer:

- **Compositional search chains.** We could implement multi-step retrieval where the model first finds candidate evidence, extracts useful entities or terms, and then launches follow-up searches based on those intermediate clues. This would better match the DCI idea of iterative hypothesis refinement.

- **Evidence localization metrics.** Beyond final accuracy, we could measure whether the system actually found the relevant document or passage that supports the answer. This would help separate cases where the model guessed correctly from cases where it retrieved and used valid evidence.

- **Context-management strategies.** The DCI paper discusses truncation, compaction, and summarization to prevent long search trajectories from overwhelming the model context. We would have liked to test similar strategies for our live-search logs, especially when retrieved pages are long or noisy.

Overall, the DCI paper helped frame our project as more than a model benchmark. Our experiments compare model accuracy, quantization, throughput, live retrieval, ASR, and confidence estimation, but the next step would be to make the retrieval component more agentic: not just retrieving documents, but actively searching, filtering, verifying, and reusing evidence across the benchmark.

# 17. Use of AI

During the development of this project, we used AI-assisted and **agentic-enabled IDEs**, such as Google Antigravity, to speed up experimentation, debugging, and iteration. These tools were not used as a replacement for our own reasoning, but as a way to accelerate the research workflow and make it easier to test ideas quickly.

Our workflow was based on repeated experimentation. We first used the literature review to understand possible directions, then implemented experiments, evaluated the results, and identified bottlenecks or bugs. AI tools helped us move faster through this loop by assisting with code generation, debugging, refactoring, and pipeline organization.

A key part of the process was building a pipeline that allowed us to log and inspect intermediate outputs. For example, we saved CSV files containing model answers, live-search results, execution times, generated Python code, syntax errors, and other debugging information. This made it easier to understand what each component of the system was doing, where failures occurred, and how different approaches compared.

The quick deployment of experiments was enabled by this pipeline. Once the main structure was in place, we could rapidly test different models, retrieval methods, quantization strategies, and fallback mechanisms. AI-assisted development was especially useful for speeding up implementation details, while the main experimental design, interpretation of results, and final decisions were guided by our own analysis.

Overall, AI tools supported the development process by making iteration faster and helping us debug complex experiments, while the core research direction, evaluation strategy, and conclusions remained our own.

### Final VRAM Cleanup
Always clean up your VRAM when you are done to free the GPU for other tasks!

In [ ]:
del model
del tokenizer
import gc
gc.collect()
import torch
torch.cuda.empty_cache()
print("VRAM Cleared. Run complete!")